In [ ]:
DATASET = 'mm'
MODE = 'smoke'
FILES = {'conformal_px1.py': '"""\nPX1 stage 4: a per-flood guarantee on missed flooding.\n\nPhase 1 (GPU): for every held-out flood E and model variant (V0 radar only, V1 radar +\nelevation), load the leave-one-event-out model that never saw E, predict water\nprobabilities on every chip of E, and store histograms of the probability for\nlabelled-water and labelled-dry pixels (all chips, and clean chips only).\n\nPhase 2: for each E, pick a water threshold using only the OTHER floods\' curves, each\nmeasured by its own held-out model (so every calibration curve is an "unseen flood"\ncurve), then score that threshold on E.\n  default  threshold 0.5 (argmax)\n  naive    largest threshold whose mean calibration miss rate is <= alpha\n  crc      conformal risk control (Angelopoulos et al.): largest t with\n           n/(n+1) * mean_FNR(t) + 1/(n+1) <= alpha     (loss bounded by 1)\nCRC guarantees that the expected miss rate on a new, exchangeable flood is <= alpha.\nMiss rate (FNR) = labelled-water pixels predicted dry / labelled-water pixels.\n\nUsage:  python conformal_px1.py --data data --runs runs_loeo\n        python conformal_px1.py --analyse-only   (reuse runs_loeo/conformal_hist.npz)\n"""\nimport argparse\nimport json\nimport os\n\nimport numpy as np\n\nBINS = 1000\nALPHAS = [0.1, 0.2, 0.3]\nEVENTS = ["Bolivia", "Ghana", "India", "Mekong", "Nigeria", "Pakistan", "Paraguay",\n          "Somalia", "Spain", "Sri-Lanka", "USA"]\nVARIANTS = {"V0": "s1", "V1": "s1dem"}\n\n\ndef phase1(args):\n    import torch\n    from sen1floods11_unet_baseline import IGNORE, UNet, channel_stats, event_of, read_split\n    from train_px1 import Chips\n\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    with open(os.path.join("audit", "clean_chips.txt")) as f:\n        clean = {ln.strip() for ln in f if ln.strip()}\n    splits = {k: read_split(args.data, k) for k in ["train", "valid", "test", "bolivia"]}\n    every = splits["train"] + splits["valid"] + splits["test"] + splits["bolivia"]\n    hist = {}\n    for v, inputs in VARIANTS.items():\n        use_dem = inputs == "s1dem"\n        for e in EVENTS:\n            ckpt = os.path.join(args.runs, f"{v}_hold{e}_seed0", "model.pt")\n            if not os.path.exists(ckpt):\n                print(f"missing {ckpt}", flush=True)\n                continue\n            train = [p for p in splits["train"] if event_of(p[0]) != e]\n            mean, std = channel_stats(args.data, train)  # same stats the model was trained with\n            model = UNet(cin=4 if use_dem else 2).to(device)\n            model.load_state_dict(torch.load(ckpt, map_location=device))\n            model.eval()\n            ds = Chips(args.data, [p for p in every if event_of(p[0]) == e], mean, std, False, use_dem)\n            h = np.zeros((2, 2, BINS), np.int64)  # [subset all/clean][label dry/water][bin]\n            with torch.no_grad():\n                for i in range(len(ds)):\n                    x, y, _, s = ds[i]\n                    p = model(x[None].to(device)).softmax(1)[0, 1].cpu().numpy()\n                    t = y.numpy()\n                    m = t != IGNORE\n                    b = np.clip((p * BINS).astype(int), 0, BINS - 1)\n                    for sub in ([0, 1] if s in clean else [0]):\n                        for lab in (0, 1):\n                            h[sub, lab] += np.bincount(b[m & (t == lab)], minlength=BINS)\n            hist[f"{v}|{e}"] = h\n            print(f"{v} hold {e}: {len(ds)} chips, water px {h[0, 1].sum()}", flush=True)\n    np.savez_compressed(os.path.join(args.runs, "conformal_hist.npz"), **hist)\n\n\ndef curves(h):\n    """For thresholds t_k = k/BINS: predicted water iff p >= t_k. Returns FNR, precision, IoU."""\n    dry, wat = h[0].astype(float), h[1].astype(float)\n    # pixels with bin >= k are predicted water at threshold k/BINS\n    wat_pos = np.cumsum(wat[::-1])[::-1]\n    dry_pos = np.cumsum(dry[::-1])[::-1]\n    W = wat.sum()\n    tp, fp, fn = wat_pos, dry_pos, W - wat_pos\n    fnr = fn / max(W, 1)\n    prec = np.divide(tp, tp + fp, out=np.ones_like(tp), where=(tp + fp) > 0)\n    iou = np.divide(tp, tp + fp + fn, out=np.zeros_like(tp), where=(tp + fp + fn) > 0)\n    return fnr, prec, iou\n\n\ndef phase2(args):\n    data = np.load(os.path.join(args.runs, "conformal_hist.npz"))\n    out = {}\n    for v in VARIANTS:\n        for sub, sub_name in [(0, "all"), (1, "clean")]:\n            cur = {}\n            for e in EVENTS:\n                key = f"{v}|{e}"\n                if key in data and data[key][sub, 1].sum() > 0:\n                    cur[e] = curves(data[key][sub])\n            rows = []\n            for e in cur:\n                others = [cur[o][0] for o in cur if o != e]\n                n = len(others)\n                mean_fnr = np.mean(others, axis=0)  # mean calibration miss rate per threshold\n                fnr, prec, iou = cur[e]\n                k05 = BINS // 2\n                row = {"event": e, "default": {"fnr": fnr[k05], "precision": prec[k05], "iou": iou[k05]}}\n                for a in ALPHAS:\n                    ok_naive = np.where(mean_fnr <= a)[0]\n                    ok_crc = np.where(n / (n + 1) * mean_fnr + 1 / (n + 1) <= a)[0]\n                    for name, ok in [("naive", ok_naive), ("crc", ok_crc)]:\n                        k = int(ok.max()) if len(ok) else 0  # largest threshold meeting the bound\n                        row[f"{name}_{a}"] = {"threshold": k / BINS, "fnr": fnr[k], "precision": prec[k], "iou": iou[k]}\n                rows.append(row)\n            summ = {"n_events": len(rows)}\n            for m in ["default"] + [f"{nm}_{a}" for a in ALPHAS for nm in ["naive", "crc"]]:\n                f = np.array([r[m]["fnr"] for r in rows])\n                target = None if m == "default" else float(m.split("_")[1])\n                summ[m] = {"mean_fnr": round(float(f.mean()), 4), "max_fnr": round(float(f.max()), 4),\n                           "events_meeting_target": None if target is None else int((f <= target).sum()),\n                           "mean_precision": round(float(np.mean([r[m]["precision"] for r in rows])), 4),\n                           "mean_iou": round(float(np.mean([r[m]["iou"] for r in rows])), 4)}\n            for r in rows:\n                for m, d in r.items():\n                    if isinstance(d, dict):\n                        r[m] = {k: round(float(x), 4) for k, x in d.items()}\n            out[f"{v}|{sub_name}"] = {"summary": summ, "events": rows}\n    with open(os.path.join(args.runs, "conformal_results.json"), "w") as f:\n        json.dump(out, f, indent=2)\n\n    lines = []\n    for key, res in out.items():\n        s = res["summary"]\n        lines += [f"### {key} ({s[\'n_events\']} held-out floods)", "",\n                  "| Threshold rule | Mean miss rate | Worst flood | Floods meeting target | Mean precision | Mean IoU |",\n                  "| --- | --- | --- | --- | --- | --- |"]\n        for m in ["default"] + [f"{nm}_{a}" for a in ALPHAS for nm in ["naive", "crc"]]:\n            d = s[m]\n            meet = "-" if d["events_meeting_target"] is None else f"{d[\'events_meeting_target\']}/{s[\'n_events\']}"\n            lines.append(f"| {m} | {d[\'mean_fnr\']:.3f} | {d[\'max_fnr\']:.3f} | {meet} | {d[\'mean_precision\']:.3f} | {d[\'mean_iou\']:.3f} |")\n        lines.append("")\n    with open(os.path.join(args.runs, "conformal_summary.md"), "w") as f:\n        f.write("\\n".join(lines))\n    print("\\n".join(lines))\n    print("CONFORMAL_DONE")\n\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--data", default="data")\n    ap.add_argument("--runs", default="runs_loeo")\n    ap.add_argument("--analyse-only", action="store_true")\n    a = ap.parse_args()\n    if not a.analyse_only:\n        phase1(a)\n    phase2(a)\n', 'tta_px1.py': '"""\nPX1 stage 5: label-free adaptation to each unseen flood, combined with the miss-rate guarantee.\n\nFor every held-out flood E and variant (V0 radar only, V1 radar + elevation) we take the\nleave-one-event-out model that never saw E and adapt it to E\'s chips WITHOUT labels:\n\n  none   no adaptation (reproduces conformal_px1.py)\n  tent   TENT: minimise prediction entropy, update batch-norm parameters only\n  st     mean-teacher self-training (the recipe of Chen et al. 2026): EMA teacher makes\n         confident pseudo-labels (p > 0.9 water, p < 0.1 dry) on a weak view, the student\n         learns them on a strong view (speckle-like noise, +-1.5 dB radiometric shift)\n  phys   st + physics-checked pseudo-labels (ours). Rules fixed in advance (same radar\n         thresholds as the label audit):\n           water pseudo-label dropped if the pixel is radar-bright (VV > -15 dB) or\n           physically implausible for ponding (> 15 m above the chip\'s low ground, or slope > 8 deg)\n           a pixel that is radar-dark (VV < -18 and VH < -25 dB), low (< 5 m) and flat (< 3 deg)\n           becomes a water pseudo-label unless the teacher is confident it is dry (then ignored)\n         Elevation is used here only to check pseudo-labels, so it also helps the radar-only model.\n\nLabels of E are used only for scoring. Every setting is fixed before seeing any result\n(200 steps, batch 8, 256 crops, Adam; TENT lr 1e-3, self-training lr 1e-4, EMA 0.99).\n\nThen, as in conformal_px1.py, the threshold for E is calibrated on the OTHER floods, each\nadapted the same way by its own held-out model, and scored on E.\nA classical physics baseline (Otsu threshold on VV per flood, no model) is reported too.\n\nAblations (physics check with one rule only): --methods phys_drop,phys_add\n\nUsage:  python tta_px1.py --data data --runs runs_loeo     (resumes from runs_loeo/tta_hist.npz)\n        python tta_px1.py --analyse-only\n"""\nimport argparse\nimport copy\nimport json\nimport os\nimport time\n\nos.environ.setdefault("GDAL_CACHEMAX", "64")\n\nimport numpy as np\n\nfrom conformal_px1 import ALPHAS, BINS, EVENTS, VARIANTS, curves\n\nMETHODS = ["none", "tent", "st", "phys"]\nABLATIONS = ["phys_drop", "phys_add"]  # physics check with only the drop rule / only the add rule\nCOMBOS = ["adabn", "adabn_phys", "tent_phys"]  # added after the first Kuro Siwo results: recalibrate, then physics-checked self-training\nADAPTIVE = ["physa", "adabn_physa"]  # per-flood radar thresholds + HAND gating (add rule only)\nALL_METHODS = METHODS + ABLATIONS + COMBOS + ADAPTIVE\nALL_METHODS += ["tent_batch"]  # separate key: never reuse historical tent histograms\nSTEPS, BATCH, CROP = 200, 8, 256\nLR_TENT, LR_ST, EMA, P_HI, P_LO = 1e-3, 1e-4, 0.99, 0.9, 0.1\nADABN_BATCHES = 50\nVV_BRIGHT, VV_DARK, VH_DARK = -15.0, -18.0, -25.0\nRELH_HIGH, SLOPE_HIGH, RELH_LOW, SLOPE_LOW = 15.0, 8.0, 5.0, 3.0\n# adaptive rule (fixed before any run): per flood, Otsu split of all VV (and VH) pixels; "dark" = below the\n# midpoint between the dark-mode mean and the Otsu threshold, capped at -15 dB (VV) / -22 dB (VH) so that a\n# flood with little water cannot turn dry land into water; low = HAND < 5 m; flat = slope < 3 deg\nHAND_LOW, VV_CAP, VH_CAP = 5.0, -15.0, -22.0\n\n\ndef otsu_dark(values):\n    """Otsu threshold of a 1-D sample and the mean of the class below it."""\n    v = values[values > -49.9]\n    hist, edges = np.histogram(v, bins=256)\n    c = (edges[:-1] + edges[1:]) / 2\n    w0 = np.cumsum(hist); w1 = w0[-1] - w0\n    s0 = np.cumsum(hist * c)\n    mu0 = s0 / np.maximum(w0, 1); mu1 = (s0[-1] - s0) / np.maximum(w1, 1)\n    k = int(np.argmax(w0 * w1 * (mu0 - mu1) ** 2))\n    return float(c[k]), float(mu0[k])\n\n\ndef adaptive_dark(raw_list):\n    """Per-flood dark thresholds (VV, VH) from the flood\'s own backscatter, no labels."""\n    out = []\n    for ch, cap in [(0, VV_CAP), (1, VH_CAP)]:\n        v = np.concatenate([r[ch].ravel()[::7] for r in raw_list])\n        t, mu = otsu_dark(v)\n        out.append(min(cap, (t + mu) / 2))\n    return out\n\n\ndef hand_channel(hand, terr):\n    """Third terrain channel: HAND / 50 m (falls back to the chip\'s relative height if HAND is missing)."""\n    h = terr[0:1] if hand is None else np.clip(hand[None].astype(np.float32), 0, 100) / 50.0\n    return np.concatenate([terr, h], 0).astype(np.float32)\n\n\ndef load_event(args, e, every, clean):\n    from sen1floods11_unet_baseline import event_of, load_label, load_s1\n    from train_px1 import load_dem, terrain_features\n    ev = {"raw": [], "terr": [], "lab": [], "clean": []}\n    for s, l in [p for p in every if event_of(p[0]) == e]:\n        s1_path = os.path.join(args.data, "S1Hand", s)\n        ev["raw"].append(load_s1(s1_path).astype(np.float32))\n        hp = os.path.join(args.data, "HAND", s.replace("S1Hand", "HAND"))\n        hand = None\n        if os.path.exists(hp):\n            import rasterio\n            with rasterio.open(hp) as src:\n                hand = src.read(1)\n        ev["hand_ok"] = ev.get("hand_ok", 0) + (hand is not None)\n        ev["terr"].append(hand_channel(hand, terrain_features(load_dem(args.data, s, s1_path), s1_path)))\n        ev["lab"].append(load_label(os.path.join(args.data, "LabelHand", l)))\n        ev["clean"].append(s in clean)\n    return ev\n\n\ndef run_one(method, v, e, ev, mean, std, args, device, ckpt=None):\n    import torch\n    import torch.nn.functional as F\n    from sen1floods11_unet_baseline import IGNORE, UNet\n\n    use_dem = VARIANTS[v] == "s1dem"\n    torch.manual_seed(args.seed)\n    gen = np.random.default_rng(args.seed)\n    m_t = torch.tensor(mean, device=device)[None, :, None, None]\n    s_t = torch.tensor(std, device=device)[None, :, None, None]\n    raw = [torch.from_numpy(r).to(device) for r in ev["raw"]]\n    terr = [torch.from_numpy(t).to(device) for t in ev["terr"]]\n\n    def inp(r, t):\n        x = (r - m_t) / s_t\n        return torch.cat([x, t[:, :2]], 1) if use_dem else x\n\n    model = UNet(cin=4 if use_dem else 2).to(device)\n    ckpt = ckpt or os.path.join(args.runs, f"{v}_hold{e}_seed{args.model_seed}", "model.pt")\n    model.load_state_dict(torch.load(ckpt, map_location=device))\n\n    def crops():\n        rs, ts = [], []\n        for i in gen.integers(0, len(raw), BATCH):\n            H, W = raw[i].shape[1:]\n            c = min(CROP, H, W)\n            r0, c0 = gen.integers(0, H - c + 1), gen.integers(0, W - c + 1)\n            r, t = raw[i][:, r0:r0 + c, c0:c0 + c], terr[i][:, r0:r0 + c, c0:c0 + c]\n            k = int(gen.integers(0, 4))\n            r, t = torch.rot90(r, k, (1, 2)), torch.rot90(t, k, (1, 2))\n            if gen.random() < 0.5:\n                r, t = r.flip(2), t.flip(2)\n            rs.append(r); ts.append(t)\n        return torch.stack(rs), torch.stack(ts)\n\n    # optional first stage: re-estimate batch-norm statistics (adabn) or TENT, then optional self-training\n    pre, base = None, method\n    if method in ("adabn", "tent", "tent_batch"):\n        pre, base = method, "none"\n    elif method.startswith(("adabn_", "tent_")):\n        pre, base = method.split("_", 1)\n    if pre == "adabn":\n        bns = [mod for mod in model.modules() if isinstance(mod, torch.nn.BatchNorm2d)]\n        for b in bns:\n            b.reset_running_stats(); b.momentum = None  # cumulative average over the target batches\n        model.train()\n        with torch.no_grad():\n            for _ in range(ADABN_BATCHES):\n                r, t = crops()\n                model(inp(r, t))\n        for b in bns:\n            b.momentum = 0.1\n    if pre in ("tent", "tent_batch"):\n        model.train()\n        if pre == "tent_batch":\n            # Official TENT BN configuration, also retained during final.eval().\n            # Event-wise 200-step adaptation and pixel-wise entropy remain PX1 choices.\n            for mod in model.modules():\n                if isinstance(mod, torch.nn.BatchNorm2d):\n                    mod.track_running_stats = False\n                    mod.running_mean = None\n                    mod.running_var = None\n        params = [p for mod in model.modules() if isinstance(mod, torch.nn.BatchNorm2d) for p in (mod.weight, mod.bias)]\n        for p in model.parameters():\n            p.requires_grad_(False)\n        for p in params:\n            p.requires_grad_(True)\n        opt = torch.optim.Adam(params, lr=LR_TENT)\n        for _ in range(args.steps):\n            r, t = crops()\n            p = model(inp(r, t)).softmax(1)\n            loss = -(p * torch.log(p.clamp_min(1e-6))).sum(1).mean()\n            opt.zero_grad(); loss.backward(); opt.step()\n        for p in model.parameters():\n            p.requires_grad_(True)\n    final = model\n    if base in ("st", "phys", "phys_drop", "phys_add", "physa"):\n        method = base\n        if method == "physa":\n            tv, th = adaptive_dark(ev["raw"])\n            print(f"    adaptive dark thresholds VV {tv:.1f} dB, VH {th:.1f} dB; HAND for {ev.get(\'hand_ok\', 0)}/{len(raw)} chips", flush=True)\n        teacher = copy.deepcopy(model).eval()\n        for p in teacher.parameters():\n            p.requires_grad_(False)\n        model.train()\n        opt = torch.optim.Adam(model.parameters(), lr=LR_ST)\n        ce = torch.nn.CrossEntropyLoss(weight=torch.tensor([1.0, 3.0], device=device), ignore_index=IGNORE)\n        kept = []\n        for _ in range(args.steps):\n            r, t = crops()\n            with torch.no_grad():\n                pt = teacher(inp(r, t)).softmax(1)[:, 1]\n                y = torch.full_like(pt, IGNORE, dtype=torch.long)\n                y[pt > P_HI] = 1\n                y[pt < P_LO] = 0\n                relh, slope = t[:, 0] * 50, t[:, 1] * 10\n                if method in ("phys", "phys_drop"):\n                    implausible = (r[:, 0] > VV_BRIGHT) | (relh > RELH_HIGH) | (slope > SLOPE_HIGH)\n                    y[(y == 1) & implausible] = IGNORE\n                if method == "physa":\n                    hand = t[:, 2] * 50\n                    dark_low = (r[:, 0] < tv) & (r[:, 1] < th) & (hand < HAND_LOW) & (slope < SLOPE_LOW)\n                    y[dark_low & (pt >= P_LO)] = 1\n                    y[dark_low & (pt < P_LO)] = IGNORE\n                if method in ("phys", "phys_add"):\n                    dark_low = (r[:, 0] < VV_DARK) & (r[:, 1] < VH_DARK) & (relh < RELH_LOW) & (slope < SLOPE_LOW)\n                    y[dark_low & (pt >= P_LO)] = 1\n                    y[dark_low & (pt < P_LO)] = IGNORE\n                kept.append(float((y != IGNORE).float().mean()))\n                off = (torch.rand(r.shape[0], 1, 1, 1, device=device) * 3 - 1.5)\n            xs = inp(r + off, t)\n            xs = torch.cat([xs[:, :2] + 0.1 * torch.randn_like(xs[:, :2]), xs[:, 2:]], 1)\n            loss = ce(model(xs), y)\n            opt.zero_grad(); loss.backward(); opt.step()\n            with torch.no_grad():\n                for pt_, ps_ in zip(teacher.parameters(), model.parameters()):\n                    pt_.mul_(EMA).add_(ps_.detach(), alpha=1 - EMA)\n                for bt, bs in zip(teacher.buffers(), model.buffers()):\n                    if bt.dtype.is_floating_point:\n                        bt.mul_(EMA).add_(bs, alpha=1 - EMA)\n                    else:\n                        bt.copy_(bs)\n        print(f"    pseudo-labelled share of pixels {np.mean(kept):.3f}", flush=True)\n        final = teacher\n\n    final.eval()\n    h = np.zeros((2, 2, BINS), np.int64)\n    with torch.no_grad():\n        for i in range(len(raw)):\n            p = final(inp(raw[i][None], terr[i][None])).softmax(1)[0, 1]\n            b = (p * BINS).long().clamp(0, BINS - 1)\n            lab = torch.from_numpy(ev["lab"][i].astype(np.int64)).to(device)\n            for sub in ([0, 1] if ev["clean"][i] else [0]):\n                for c in (0, 1):\n                    h[sub, c] += torch.bincount(b[lab == c], minlength=BINS).cpu().numpy()\n            if "lab_alt" in ev:\n                lab = torch.from_numpy(ev["lab_alt"][i].astype(np.int64)).to(device)\n                for c in (0, 1):\n                    h[1, c] += torch.bincount(b[lab == c], minlength=BINS).cpu().numpy()\n    return h\n\n\ndef otsu_counts(ev):\n    """Classical physics baseline: one Otsu threshold on VV per flood; water = VV below it."""\n    vv = np.concatenate([r[0].ravel() for r in ev["raw"]])\n    hist, edges = np.histogram(vv[vv > -49.9], bins=256)\n    c = (edges[:-1] + edges[1:]) / 2\n    w0 = np.cumsum(hist); w1 = w0[-1] - w0\n    mu0 = np.cumsum(hist * c) / np.maximum(w0, 1)\n    mu1 = (np.sum(hist * c) - np.cumsum(hist * c)) / np.maximum(w1, 1)\n    thr = c[np.argmax(w0 * w1 * (mu0 - mu1) ** 2)]\n    out = np.zeros((2, 3), np.int64)  # [all/clean][tp, fp, fn]\n    for r, lab, cl in zip(ev["raw"], ev["lab"], ev["clean"]):\n        pred = r[0] < thr\n        cnt = [np.sum(pred & (lab == 1)), np.sum(pred & (lab == 0)), np.sum(~pred & (lab == 1))]\n        for sub in ([0, 1] if cl else [0]):\n            out[sub] += cnt\n    return out, float(thr)\n\n\ndef save(path, d):\n    tmp = path[:-4] + ".tmp.npz"\n    np.savez_compressed(tmp, **d)\n    os.replace(tmp, path)\n\n\ndef phase1(args):\n    import torch\n    from sen1floods11_unet_baseline import channel_stats, event_of, read_split\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    path = os.path.join(args.runs, f"tta_hist{tag(args)}.npz")\n    done = dict(np.load(path)) if os.path.exists(path) else {}\n    with open(os.path.join("audit", "clean_chips.txt")) as f:\n        clean = {ln.strip() for ln in f if ln.strip()}\n    splits = {k: read_split(args.data, k) for k in ["train", "valid", "test", "bolivia"]}\n    every = splits["train"] + splits["valid"] + splits["test"] + splits["bolivia"]\n    for e in EVENTS:\n        todo = [(m, v) for v in VARIANTS for m in args.methods\n                if f"{m}|{v}|{e}" not in done and os.path.exists(os.path.join(args.runs, f"{v}_hold{e}_seed{args.model_seed}", "model.pt"))]\n        if not todo and f"otsu|{e}" in done:\n            continue\n        ev = load_event(args, e, every, clean)\n        if not ev["raw"]:\n            continue\n        mean, std = channel_stats(args.data, [p for p in splits["train"] if event_of(p[0]) != e])\n        if f"otsu|{e}" not in done:\n            done[f"otsu|{e}"], thr = otsu_counts(ev)\n            print(f"{e}: Otsu VV threshold {thr:.1f} dB", flush=True)\n        for m, v in todo:\n            t0 = time.time()\n            h = run_one(m, v, e, ev, mean, std, args, device)\n            done[f"{m}|{v}|{e}"] = h\n            fnr, _, iou = curves(h[0])\n            print(f"{m:9s} {v} hold {e:9s}: miss rate at 0.5 {fnr[BINS // 2]:.3f}, IoU {iou[BINS // 2]:.3f}, "\n                  f"{time.time() - t0:.0f}s", flush=True)\n            save(path, done)\n        save(path, done)\n        del ev\n        if device == "cuda":\n            torch.cuda.empty_cache()\n\n\ndef tag(args):\n    return "" if args.seed == 0 and args.model_seed == 0 else f"_m{args.model_seed}_a{args.seed}"\n\n\ndef compute(data, events=None):\n    events = events or EVENTS\n    out = {}\n    k05 = BINS // 2\n    for v in VARIANTS:\n        for sub, sub_name in [(0, "all"), (1, "clean")]:\n            block = {}\n            for m in [m for m in ALL_METHODS if any(k.startswith(m + "|") for k in data)]:\n                cur = {e: curves(data[f"{m}|{v}|{e}"][sub]) for e in events\n                       if f"{m}|{v}|{e}" in data and data[f"{m}|{v}|{e}"][sub, 1].sum() > 0}\n                if len(cur) < 2:\n                    continue\n                per = {}\n                for e in cur:\n                    others = np.mean([cur[o][0] for o in cur if o != e], axis=0)\n                    n = len(cur) - 1\n                    fnr, prec, iou = cur[e]\n                    per[e] = {"default": {"fnr": fnr[k05], "iou": iou[k05], "precision": prec[k05]}}\n                    for a in ALPHAS:\n                        ok = np.where(n / (n + 1) * others + 1 / (n + 1) <= a)[0]\n                        k = int(ok.max()) if len(ok) else 0\n                        per[e][f"crc_{a}"] = {"threshold": k / BINS, "fnr": fnr[k], "iou": iou[k], "precision": prec[k]}\n                summ = {}\n                for rule in ["default"] + [f"crc_{a}" for a in ALPHAS]:\n                    f = np.array([per[e][rule]["fnr"] for e in per])\n                    a = None if rule == "default" else float(rule.split("_")[1])\n                    summ[rule] = {"mean_fnr": float(f.mean()), "max_fnr": float(f.max()), "sd_fnr": float(f.std()),\n                                  "meeting": None if a is None else int((f <= a).sum()),\n                                  "mean_iou": float(np.mean([per[e][rule]["iou"] for e in per]))}\n                block[m] = {"n": len(per), "summary": summ,\n                            "events": {e: {r: {k: round(float(x), 4) for k, x in d.items()} for r, d in pe.items()} for e, pe in per.items()}}\n            if True:\n                ot = [data[f"otsu|{e}"][sub] for e in events if f"otsu|{e}" in data]\n                if ot:\n                    ious = [c[0] / max(1, c.sum()) for c in ot]\n                    fnrs = [c[2] / max(1, c[0] + c[2]) for c in ot]\n                    block["otsu"] = {"n": len(ot), "summary": {"default": {"mean_fnr": float(np.mean(fnrs)), "max_fnr": float(np.max(fnrs)),\n                                     "sd_fnr": float(np.std(fnrs)), "meeting": None, "mean_iou": float(np.mean(ious))}}}\n            out[f"{v}|{sub_name}"] = block\n    return out\n\n\ndef analyse(args):\n    data = dict(np.load(os.path.join(args.runs, f"tta_hist{tag(args)}.npz")))\n    report(compute(data), os.path.join(args.runs, f"tta_results{tag(args)}.json"),\n           os.path.join(args.runs, f"tta_summary{tag(args)}.md"))\n\n\ndef report(out, json_path, md_path):\n    lines = []\n    for key, block in out.items():\n            v, sub_name = key.split("|")\n            lines += [f"### {v}, {sub_name} chips", "",\n                      "| Adaptation | Rule | Mean miss rate | Worst flood | SD across floods | Floods meeting target | Mean IoU |",\n                      "| --- | --- | --- | --- | --- | --- | --- |"]\n            for m, b in block.items():\n                for rule, d in b["summary"].items():\n                    meet = "-" if d["meeting"] is None else f"{d[\'meeting\']}/{b[\'n\']}"\n                    lines.append(f"| {m} | {rule} | {d[\'mean_fnr\']:.3f} | {d[\'max_fnr\']:.3f} | {d[\'sd_fnr\']:.3f} | {meet} | {d[\'mean_iou\']:.3f} |")\n            lines.append("")\n            if "phys" in block and "none" in block:\n                lines += [f"Per flood at the 20% target ({v}, {sub_name}): miss rate / IoU", "",\n                          "| Flood | " + " | ".join(m for m in ALL_METHODS if m in block) + " |",\n                          "| --- |" + " --- |" * len([m for m in ALL_METHODS if m in block])]\n                for e in block["none"]["events"]:\n                    cells = []\n                    for m in ALL_METHODS:\n                        if m in block and e in block[m]["events"]:\n                            d = block[m]["events"][e]["crc_0.2"]\n                            cells.append(f"{d[\'fnr\']:.3f} / {d[\'iou\']:.3f}")\n                        elif m in block:\n                            cells.append("-")\n                    lines.append(f"| {e} | " + " | ".join(cells) + " |")\n                lines.append("")\n    with open(json_path, "w") as f:\n        json.dump(out, f, indent=1)\n    with open(md_path, "w") as f:\n        f.write("\\n".join(lines))\n    print("\\n".join(lines))\n    print("TTA_DONE")\n\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--data", default="data")\n    ap.add_argument("--runs", default="runs_loeo")\n    ap.add_argument("--analyse-only", action="store_true")\n    ap.add_argument("--steps", type=int, default=STEPS)\n    ap.add_argument("--seed", type=int, default=0, help="adaptation seed")\n    ap.add_argument("--model-seed", type=int, default=0, help="seed of the leave-one-event-out models to adapt")\n    ap.add_argument("--methods", default=",".join(METHODS), help="comma-separated, from " + ",".join(ALL_METHODS))\n    ap.add_argument("--no-analyse", action="store_true")\n    a = ap.parse_args()\n    a.methods = a.methods.split(",")\n    if not a.analyse_only:\n        phase1(a)\n    if not a.no_analyse:\n        analyse(a)\n', 'ks_eval.py': '"""\nPX1 stage 6: cross-dataset test on Kuro Siwo (up to 43 flood events, none seen in training).\n\nModels: the Sen1Floods11 models trained on the standard training split (runs/V0_seed*, V1_seed*).\nEvery Kuro Siwo flood event is unseen. For each event we apply the same label-free\nadaptation methods as tta_px1.py (none, tent, st, phys) and store probability histograms;\nthen the miss-rate guarantee is calibrated leave-one-event-out across the Kuro Siwo events,\nso with ~40 events the guarantee is far less conservative than with 11.\n\nInputs per chip: post-flood VV, VH (dB) and the Kuro Siwo DEM (10 m grid). Labels: permanent\nwater and flood water both count as water; class 3 and invalid pixels are ignored.\n\nUsage:  python ks_eval.py --data data --ks data_ks --runs runs --model-seed 0 --out runs_ks\n        python ks_eval.py --analyse-only --model-seed 0\n"""\nimport argparse\nimport csv\nimport glob\nimport os\nimport time\n\nos.environ.setdefault("GDAL_CACHEMAX", "64")\n\nimport numpy as np\n\nfrom tta_px1 import BINS, COMBOS, METHODS, VARIANTS, hand_channel, compute, curves, otsu_counts, report, run_one, save\n\nMIN_CHIPS = 8\nV = "_v2"  # v2: events merged across Kuro Siwo splits (v1 dropped the test chips of events in both)\n\n\ndef terrain_10m(dem):\n    gy, gx = np.gradient(dem.astype(np.float64), 10.0, 10.0)\n    slope = np.degrees(np.arctan(np.hypot(gx, gy)))\n    relh = dem - np.percentile(dem, 2)\n    return np.stack([np.clip(relh / 50.0, 0, 2), np.clip(slope / 10.0, 0, 3)]).astype(np.float32)\n\n\ndef load_ks_event(files):\n    ev = {"raw": [], "terr": [], "lab": [], "clean": []}\n    for f in files:\n        z = np.load(f)\n        raw = np.clip(np.stack([z["vv"], z["vh"]]).astype(np.float32), -50, 5)\n        dem = z["dem"].astype(np.float32)\n        if not np.all(np.isfinite(dem)):\n            dem = np.nan_to_num(dem, nan=float(np.nanmedian(dem)) if np.isfinite(dem).any() else 0.0)\n        m = z["mask"]\n        lab = np.full(m.shape, 255, np.uint8)\n        lab[m == 0] = 0\n        lab[(m == 1) | (m == 2)] = 1\n        lab[(z["valid"] == 0) | ~np.isfinite(z["vv"]) | ~np.isfinite(z["vh"])] = 255\n        raw = np.nan_to_num(raw, nan=-50.0)\n        ev["raw"].append(raw)\n        hp = f.replace("data_ks", "data_ks_hand", 1)[:-4] + ".npy"\n        hand = np.load(hp) if os.path.exists(hp) else None\n        ev["hand_ok"] = ev.get("hand_ok", 0) + (hand is not None)\n        ev["terr"].append(hand_channel(hand, terrain_10m(dem)))\n        ev["lab"].append(lab)\n        ev["clean"].append(False)\n    return ev\n\n\ndef events_of(ks):\n    """Event id -> chip files, merged across the test_GRD and train_GRD folders (an event can be in both)."""\n    ev = {}\n    for d in sorted(glob.glob(os.path.join(ks, "*_GRD", "*"))):\n        ev.setdefault(os.path.basename(d), []).extend(sorted(glob.glob(os.path.join(d, "*.npz"))))\n    return {e: f for e, f in sorted(ev.items()) if len(f) >= MIN_CHIPS}\n\n\ndef phase1(args):\n    import torch\n    from sen1floods11_unet_baseline import channel_stats, read_split\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    os.makedirs(args.out, exist_ok=True)\n    path = os.path.join(args.out, f"ks_hist_m{args.model_seed}{V}.npz")\n    done = dict(np.load(path)) if os.path.exists(path) else {}\n    mean, std = channel_stats(args.data, read_split(args.data, "train"))\n    events = events_of(args.ks)\n    print(f"{len(events)} Kuro Siwo events with >= {MIN_CHIPS} chips", flush=True)\n    for e, files in events.items():\n        todo = [(m, v) for v in VARIANTS for m in args.methods if f"{m}|{v}|{e}" not in done]\n        if not todo and f"otsu|{e}" in done:\n            continue\n        ev = load_ks_event(files)\n        if f"otsu|{e}" not in done:\n            done[f"otsu|{e}"], thr = otsu_counts(ev)\n        for m, v in todo:\n            t0 = time.time()\n            ckpt = os.path.join(args.runs, f"{v}_seed{args.model_seed}", "model.pt")\n            h = run_one(m, v, e, ev, mean, std, args, device, ckpt=ckpt)\n            done[f"{m}|{v}|{e}"] = h\n            fnr, _, iou = curves(h[0])\n            print(f"{m:10s} {v} event {e:>4s} ({len(files)} chips): miss rate at 0.5 {fnr[BINS // 2]:.3f}, "\n                  f"IoU {iou[BINS // 2]:.3f}, {time.time() - t0:.0f}s", flush=True)\n            save(path, done)\n        save(path, done)\n        if device == "cuda":\n            torch.cuda.empty_cache()\n\n\ndef analyse(args):\n    data = dict(np.load(os.path.join(args.out, f"ks_hist_m{args.model_seed}{V}.npz")))\n    events = sorted({k.split("|")[2] for k in data if k.count("|") == 2})\n    report(compute(data, events), os.path.join(args.out, f"ks_results_m{args.model_seed}{V}.json"),\n           os.path.join(args.out, f"ks_summary_m{args.model_seed}{V}.md"))\n    print("KS_EVAL_DONE", flush=True)\n\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--data", default="data")\n    ap.add_argument("--ks", default="data_ks")\n    ap.add_argument("--runs", default="runs")\n    ap.add_argument("--out", default="runs_ks")\n    ap.add_argument("--model-seed", type=int, default=0)\n    ap.add_argument("--seed", type=int, default=0)\n    ap.add_argument("--steps", type=int, default=200)\n    ap.add_argument("--analyse-only", action="store_true")\n    ap.add_argument("--methods", default=",".join(METHODS + COMBOS))\n    a = ap.parse_args()\n    a.methods = a.methods.split(",")\n    if not a.analyse_only:\n        phase1(a)\n    analyse(a)\n', 'sen1floods11_unet_baseline.py': '"""\nPX1 step 3: U-Net baseline on Sen1Floods11 (Sentinel-1 only, hand-labelled split).\n\nReports water IoU on the official test split and on the held-out Bolivia event,\nplus IoU per flood event, so it doubles as the first "unseen event" measurement.\n\n1) Get the data (about 1 GB; needs the Google Cloud SDK, no account needed):\n     mkdir -p data && cd data\n     gsutil -m cp -r gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand .\n     gsutil -m cp -r gs://sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand .\n     gsutil -m cp -r gs://sen1floods11/v1.1/splits/flood_handlabeled .\n     cd ..\n2) Install:  pip install torch rasterio numpy\n3) Run:      python sen1floods11_unet_baseline.py --data data --epochs 60\n   Quick check: add  --epochs 2 --limit 20\n\nLabels: -1 = no data (ignored), 0 = not water, 1 = water.\n"""\nimport argparse\nimport csv\nimport json\nimport os\nimport random\nimport time\nfrom collections import defaultdict\n\nimport numpy as np\nimport rasterio\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.data import DataLoader, Dataset\n\nIGNORE = 255\n\n\n# ---------------------------------------------------------------- data\ndef read_split(data_dir, name):\n    path = os.path.join(data_dir, "flood_handlabeled", f"flood_{name}_data.csv")\n    with open(path) as f:\n        rows = [r for r in csv.reader(f) if r]\n    return [(r[0].strip(), r[1].strip()) for r in rows]\n\n\ndef event_of(fname):\n    return fname.split("_")[0]  # e.g. "Bolivia_103757_S1Hand.tif" -> "Bolivia"\n\n\ndef load_s1(path):\n    with rasterio.open(path) as src:\n        x = src.read().astype(np.float32)  # (2, H, W): VV, VH in dB\n    x = np.nan_to_num(x, nan=-50.0, posinf=0.0, neginf=-50.0)\n    return np.clip(x, -50.0, 5.0)\n\n\ndef load_label(path):\n    with rasterio.open(path) as src:\n        y = src.read(1).astype(np.int64)\n    y[y < 0] = IGNORE\n    return y\n\n\nclass Chips(Dataset):\n    def __init__(self, data_dir, pairs, mean, std, train):\n        self.items = [(os.path.join(data_dir, "S1Hand", s), os.path.join(data_dir, "LabelHand", l), s)\n                      for s, l in pairs]\n        self.mean = mean[:, None, None]\n        self.std = std[:, None, None]\n        self.train = train\n\n    def __len__(self):\n        return len(self.items)\n\n    def __getitem__(self, i):\n        s_path, l_path, name = self.items[i]\n        x = (load_s1(s_path) - self.mean) / self.std\n        y = load_label(l_path)\n        if self.train:\n            r, c = random.randint(0, x.shape[1] - 256), random.randint(0, x.shape[2] - 256)\n            x, y = x[:, r:r + 256, c:c + 256], y[r:r + 256, c:c + 256]\n            if random.random() < 0.5:\n                x, y = x[:, :, ::-1], y[:, ::-1]\n            if random.random() < 0.5:\n                x, y = x[:, ::-1, :], y[::-1, :]\n            k = random.randint(0, 3)\n            x, y = np.rot90(x, k, (1, 2)), np.rot90(y, k)\n        return torch.from_numpy(x.copy()), torch.from_numpy(y.copy()), name\n\n\ndef channel_stats(data_dir, pairs):\n    vals = [load_s1(os.path.join(data_dir, "S1Hand", s)).reshape(2, -1) for s, _ in pairs]\n    v = np.concatenate(vals, axis=1)\n    return v.mean(1).astype(np.float32), v.std(1).astype(np.float32) + 1e-6\n\n\n# ---------------------------------------------------------------- model\ndef block(cin, cout):\n    return nn.Sequential(\n        nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),\n        nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))\n\n\nclass UNet(nn.Module):\n    def __init__(self, cin=2, ncls=2, w=32):\n        super().__init__()\n        ch = [w, w * 2, w * 4, w * 8, w * 16]\n        self.downs = nn.ModuleList([block(cin, ch[0])] + [block(ch[i], ch[i + 1]) for i in range(4)])\n        self.ups = nn.ModuleList([nn.ConvTranspose2d(ch[i + 1], ch[i], 2, stride=2) for i in reversed(range(4))])\n        self.decs = nn.ModuleList([block(ch[i] * 2, ch[i]) for i in reversed(range(4))])\n        self.head = nn.Conv2d(ch[0], ncls, 1)\n\n    def forward(self, x):\n        skips = []\n        for i, d in enumerate(self.downs):\n            x = d(x if i == 0 else F.max_pool2d(x, 2))\n            skips.append(x)\n        x = skips.pop()\n        for up, dec in zip(self.ups, self.decs):\n            x = dec(torch.cat([up(x), skips.pop()], 1))\n        return self.head(x)\n\n\n# ---------------------------------------------------------------- evaluation\n@torch.no_grad()\ndef evaluate(model, loader, device):\n    """Water IoU pooled over all pixels, plus pooled IoU per flood event."""\n    model.eval()\n    tot = np.zeros(3)  # intersection, union, labelled pixels\n    per_event = defaultdict(lambda: np.zeros(3))\n    for x, y, names in loader:\n        pred = model(x.to(device)).argmax(1).cpu().numpy()\n        y = y.numpy()\n        for p, t, n in zip(pred, y, names):\n            m = t != IGNORE\n            inter = np.sum((p == 1) & (t == 1) & m)\n            union = np.sum(((p == 1) | (t == 1)) & m)\n            stats = np.array([inter, union, m.sum()])\n            tot += stats\n            per_event[event_of(n)] += stats\n    iou = lambda s: float(s[0] / s[1]) if s[1] > 0 else float("nan")\n    return iou(tot), {e: round(iou(s), 4) for e, s in sorted(per_event.items())}\n\n\n# ---------------------------------------------------------------- main\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--data", default="data")\n    ap.add_argument("--epochs", type=int, default=60)\n    ap.add_argument("--batch", type=int, default=8)\n    ap.add_argument("--lr", type=float, default=1e-3)\n    ap.add_argument("--seed", type=int, default=0)\n    ap.add_argument("--limit", type=int, default=0, help="use only N chips per split (smoke test)")\n    ap.add_argument("--out", default="results")\n    args = ap.parse_args()\n\n    random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    splits = {k: read_split(args.data, k) for k in ["train", "valid", "test", "bolivia"]}\n    if args.limit:\n        splits = {k: v[:args.limit] for k, v in splits.items()}\n    print({k: len(v) for k, v in splits.items()}, "device:", device)\n\n    mean, std = channel_stats(args.data, splits["train"])\n    mk = lambda k, tr, bs: DataLoader(Chips(args.data, splits[k], mean, std, tr), batch_size=bs,\n                                      shuffle=tr, num_workers=2, drop_last=tr)\n    train_dl = mk("train", True, args.batch)\n    eval_dl = {k: mk(k, False, 4) for k in ["valid", "test", "bolivia"]}\n\n    model = UNet().to(device)\n    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)\n    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=args.epochs)\n    # water is the minority class; weight it up\n    loss_fn = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 3.0], device=device), ignore_index=IGNORE)\n\n    os.makedirs(args.out, exist_ok=True)\n    best, best_path = -1.0, os.path.join(args.out, f"unet_seed{args.seed}.pt")\n    for ep in range(args.epochs):\n        model.train(); t0 = time.time(); run = 0.0\n        for x, y, _ in train_dl:\n            x, y = x.to(device), y.to(device)\n            opt.zero_grad()\n            loss = loss_fn(model(x), y)\n            loss.backward(); opt.step(); run += loss.item()\n        sched.step()\n        v_iou, _ = evaluate(model, eval_dl["valid"], device)\n        if v_iou > best:\n            best = v_iou; torch.save(model.state_dict(), best_path)\n        print(f"epoch {ep + 1:3d}  loss {run / max(1, len(train_dl)):.4f}  valid IoU {v_iou:.4f}  {time.time() - t0:.0f}s")\n\n    model.load_state_dict(torch.load(best_path, map_location=device))\n    res = {"seed": args.seed, "best_valid_iou": round(best, 4)}\n    for k in ["test", "bolivia"]:\n        iou, per = evaluate(model, eval_dl[k], device)\n        res[f"{k}_iou"] = round(iou, 4); res[f"{k}_iou_per_event"] = per\n    print(json.dumps(res, indent=2))\n    with open(os.path.join(args.out, f"results_seed{args.seed}.json"), "w") as f:\n        json.dump(res, f, indent=2)\n\n\nif __name__ == "__main__":\n    main()\n', 'train_px1.py': '"""\nPX1 stage 3: does elevation help, as an input and as a soft physics loss?\n\nVariants (pick with flags):\n  V0   radar only                              --inputs s1\n  V0c  radar only, trained on clean chips      --inputs s1 --train-subset clean\n  V1   radar + elevation features              --inputs s1dem\n  V2   radar + elevation + soft gravity loss   --inputs s1dem --phys 0.1\n\nElevation features (from the cached Copernicus DEM): height above the chip\'s low\nground (2nd percentile), divided by 50 m and clipped to [0, 2], and slope in degrees,\ndivided by 10 and clipped to [0, 3].\n\nGravity loss (soft version of EvaNet\'s elevation loss): for neighbouring pixels i, j\nwhose elevations differ by more than 0.2 m, with j lower, penalise p_i * (1 - p_j):\nwater on the higher pixel but not on the lower one. Off for 5 epochs, then ramps up\nto full weight by epoch 15.\n\nEvaluates on test and Bolivia, on all chips and on clean chips (audit/clean_chips.txt),\npooled and per event. Same U-Net, crops, augmentation and schedule as the baseline.\n\nUsage:  python train_px1.py --variant V1 --inputs s1dem --seed 0\n"""\nimport argparse\nimport json\nimport math\nimport os\nimport random\nimport time\nfrom collections import defaultdict\n\nos.environ.setdefault("GDAL_CACHEMAX", "64")  # cap GDAL\'s block cache (MB)\n\nimport numpy as np\nimport rasterio\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.data import DataLoader, Dataset\n\nfrom sen1floods11_unet_baseline import IGNORE, UNet, channel_stats, event_of, load_label, load_s1, read_split\n\nDZ_MIN = 0.2\n\n\ndef process_mem_mb():\n    """Resident memory of this process in MB (psutil if available)."""\n    try:\n        import psutil\n        return int(psutil.Process().memory_info().rss / 2**20)\n    except Exception:\n        return -1\n\n\ndef dem_path(data_dir, s):\n    return os.path.join(data_dir, "DEM", s.replace("S1Hand", "DEM"))\n\n\ndef load_dem(data_dir, s, s1_path):\n    p = dem_path(data_dir, s)\n    if not os.path.exists(p):\n        return np.zeros((512, 512), np.float32)\n    with rasterio.open(p) as src:\n        return src.read(1).astype(np.float32)\n\n\ndef terrain_features(dem, s1_path):\n    with rasterio.open(s1_path) as ref:\n        t, crs, b = ref.transform, ref.crs, ref.bounds\n    if crs.is_geographic:\n        lat = math.radians((b.top + b.bottom) / 2)\n        dx, dy = abs(t.a) * 111320 * math.cos(lat), abs(t.e) * 110540\n    else:\n        dx, dy = abs(t.a), abs(t.e)\n    gy, gx = np.gradient(dem, dy, dx)\n    slope = np.degrees(np.arctan(np.hypot(gx, gy)))\n    relh = dem - np.percentile(dem, 2)\n    return np.stack([np.clip(relh / 50.0, 0, 2), np.clip(slope / 10.0, 0, 3)]).astype(np.float32)\n\n\nclass Chips(Dataset):\n    def __init__(self, data_dir, pairs, mean, std, train, use_dem):\n        self.data_dir, self.pairs, self.train, self.use_dem = data_dir, pairs, train, use_dem\n        self.mean, self.std = mean[:, None, None], std[:, None, None]\n\n    def __len__(self):\n        return len(self.pairs)\n\n    def __getitem__(self, i):\n        s, l = self.pairs[i]\n        s1_path = os.path.join(self.data_dir, "S1Hand", s)\n        x = (load_s1(s1_path) - self.mean) / self.std\n        y = load_label(os.path.join(self.data_dir, "LabelHand", l))\n        dem = load_dem(self.data_dir, s, s1_path)\n        if self.use_dem:\n            x = np.concatenate([x, terrain_features(dem, s1_path)], 0)\n        if self.train:\n            r, c = random.randint(0, 256), random.randint(0, 256)\n            x, y, dem = x[:, r:r + 256, c:c + 256], y[r:r + 256, c:c + 256], dem[r:r + 256, c:c + 256]\n            if random.random() < 0.5:\n                x, y, dem = x[:, :, ::-1], y[:, ::-1], dem[:, ::-1]\n            if random.random() < 0.5:\n                x, y, dem = x[:, ::-1, :], y[::-1, :], dem[::-1, :]\n            k = random.randint(0, 3)\n            x, y, dem = np.rot90(x, k, (1, 2)), np.rot90(y, k), np.rot90(dem, k)\n        return (torch.from_numpy(x.copy()), torch.from_numpy(y.copy()),\n                torch.from_numpy(dem.copy()), s)\n\n\ndef gravity_loss(logits, dem):\n    p = logits.softmax(1)[:, 1]\n    total, n = logits.new_zeros(()), 0\n    for dz, pa, pb in [(dem[:, :, 1:] - dem[:, :, :-1], p[:, :, :-1], p[:, :, 1:]),\n                       (dem[:, 1:, :] - dem[:, :-1, :], p[:, :-1, :], p[:, 1:, :])]:\n        b_lower = dz < -DZ_MIN   # pixel b lower than a\n        a_lower = dz > DZ_MIN\n        total = total + (pa * (1 - pb))[b_lower].sum() + (pb * (1 - pa))[a_lower].sum()\n        n += int(b_lower.sum() + a_lower.sum())\n    return total / max(1, n)\n\n\n@torch.no_grad()\ndef evaluate(model, data_dir, pairs, mean, std, use_dem, device, clean):\n    model.eval()\n    ds = Chips(data_dir, pairs, mean, std, False, use_dem)\n    pooled = {"all": np.zeros(3), "clean": np.zeros(3)}\n    per_event = defaultdict(lambda: {"all": np.zeros(3), "clean": np.zeros(3)})\n    for i in range(len(ds)):\n        x, y, _, s = ds[i]\n        pred = model(x[None].to(device)).argmax(1)[0].cpu().numpy()\n        t = y.numpy()\n        m = t != IGNORE\n        c = np.array([np.sum((pred == 1) & (t == 1) & m), np.sum((pred == 1) & (t == 0) & m), np.sum((pred == 0) & (t == 1) & m)])\n        keys = ["all"] + (["clean"] if s in clean else [])\n        for k in keys:\n            pooled[k] += c\n            per_event[event_of(s)][k] += c\n    iou = lambda c: round(float(c[0] / c.sum()), 4) if c.sum() else None\n    return ({k: iou(v) for k, v in pooled.items()},\n            {e: {k: iou(v) for k, v in d.items()} for e, d in sorted(per_event.items())})\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--data", default="data")\n    ap.add_argument("--variant", required=True)\n    ap.add_argument("--inputs", choices=["s1", "s1dem"], default="s1")\n    ap.add_argument("--phys", type=float, default=0.0)\n    ap.add_argument("--train-subset", choices=["all", "clean"], default="all")\n    ap.add_argument("--clean-list", default=os.path.join("audit", "clean_chips.txt"))\n    ap.add_argument("--epochs", type=int, default=60)\n    ap.add_argument("--batch", type=int, default=8)\n    ap.add_argument("--lr", type=float, default=1e-3)\n    ap.add_argument("--seed", type=int, default=0)\n    ap.add_argument("--limit", type=int, default=0)\n    ap.add_argument("--workers", type=int, default=0, help="DataLoader worker processes (0 = load in main process; saves RAM on Windows)")\n    ap.add_argument("--out", default="runs")\n    ap.add_argument("--holdout", default="", help="flood event to hold out entirely (leave-one-event-out)")\n    args = ap.parse_args()\n\n    random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    use_dem = args.inputs == "s1dem"\n    with open(args.clean_list) as f:\n        clean = {ln.strip() for ln in f if ln.strip()}\n    splits = {k: read_split(args.data, k) for k in ["train", "valid", "test", "bolivia"]}\n    if args.limit:\n        splits = {k: v[:args.limit] for k, v in splits.items()}\n    if args.holdout:  # train and select on the other events; test on every chip of the held-out event\n        every = splits["train"] + splits["valid"] + splits["test"] + splits["bolivia"]\n        splits["holdout"] = [p for p in every if event_of(p[0]) == args.holdout]\n        for k in ["train", "valid"]:\n            splits[k] = [p for p in splits[k] if event_of(p[0]) != args.holdout]\n        if not splits["holdout"]:\n            raise SystemExit(f"no chips for event {args.holdout}")\n    train_pairs = splits["train"] if args.train_subset == "all" else [p for p in splits["train"] if p[0] in clean]\n    print(f"{args.variant} seed {args.seed}: {len(train_pairs)} training chips, inputs={args.inputs}, phys={args.phys}, device={device}", flush=True)\n\n    mean, std = channel_stats(args.data, splits["train"])\n    dl = DataLoader(Chips(args.data, train_pairs, mean, std, True, use_dem), batch_size=args.batch,\n                    shuffle=True, num_workers=args.workers, drop_last=True)\n    model = UNet(cin=4 if use_dem else 2).to(device)\n    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)\n    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=args.epochs)\n    ce = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 3.0], device=device), ignore_index=IGNORE)\n\n    tag = f"_hold{args.holdout}" if args.holdout else ""\n    run_dir = os.path.join(args.out, f"{args.variant}{tag}_seed{args.seed}")\n    os.makedirs(run_dir, exist_ok=True)\n    best, ckpt, last = -1.0, os.path.join(run_dir, "model.pt"), os.path.join(run_dir, "last.pt")\n    start = 0\n    if os.path.exists(last):  # resume after a crash\n        st = torch.load(last, map_location=device, weights_only=False)\n        model.load_state_dict(st["model"]); opt.load_state_dict(st["opt"]); sched.load_state_dict(st["sched"])\n        start, best = st["epoch"], st["best"]\n        random.setstate(st["py_rng"]); np.random.set_state(st["np_rng"]); torch.set_rng_state(st["torch_rng"])\n        print(f"resumed from epoch {start}, best valid IoU {best:.4f}", flush=True)\n    for ep in range(start, args.epochs):\n        model.train(); t0 = time.time(); tot = phys_tot = 0.0\n        lam = args.phys * min(1.0, max(0.0, (ep - 5) / 10))\n        for x, y, dem, _ in dl:\n            x, y, dem = x.to(device), y.to(device), dem.to(device)\n            opt.zero_grad()\n            logits = model(x)\n            loss = ce(logits, y)\n            if lam > 0:\n                g = gravity_loss(logits, dem)\n                loss = loss + lam * g\n                phys_tot += g.item()\n            loss.backward(); opt.step(); tot += loss.item()\n        sched.step()\n        v, _ = evaluate(model, args.data, splits["valid"], mean, std, use_dem, device, clean)\n        if v["all"] > best:\n            best = v["all"]; torch.save(model.state_dict(), ckpt)\n        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),\n                    "epoch": ep + 1, "best": best, "py_rng": random.getstate(), "np_rng": np.random.get_state(),\n                    "torch_rng": torch.get_rng_state()}, last)\n        print(f"epoch {ep + 1:3d} loss {tot / max(1, len(dl)):.4f} phys {phys_tot / max(1, len(dl)):.4f} "\n              f"valid IoU {v[\'all\']:.4f} {time.time() - t0:.0f}s mem {process_mem_mb()} MB", flush=True)\n\n    model.load_state_dict(torch.load(ckpt, map_location=device))\n    res = {"variant": args.variant, "seed": args.seed, "inputs": args.inputs, "phys": args.phys,\n           "train_subset": args.train_subset, "train_chips": len(train_pairs), "best_valid_iou": round(best, 4),\n           "holdout": args.holdout or None, "holdout_chips": len(splits.get("holdout", []))}\n    for k in (["holdout"] if args.holdout else ["test", "bolivia"]):\n        pooled, per = evaluate(model, args.data, splits[k], mean, std, use_dem, device, clean)\n        res[f"{k}_iou"] = pooled; res[f"{k}_per_event"] = per\n    with open(os.path.join(run_dir, "results.json"), "w") as f:\n        json.dump(res, f, indent=2)\n    print(json.dumps({k: v for k, v in res.items() if "per_event" not in k}), flush=True)\n    print("RUN_DONE", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'download_sen1floods11.py': '"""\nDownloads the hand-labelled Sen1Floods11 subset (Sentinel-1 chips, labels, splits)\nfrom the public Google Cloud bucket over plain HTTPS. Standard library only.\n\nUsage:  python download_sen1floods11.py --out data\n"""\nimport argparse\nimport json\nimport os\nimport socket\nimport sys\nimport time\nimport urllib.parse\nimport urllib.request\nfrom concurrent.futures import ThreadPoolExecutor\n\nsocket.setdefaulttimeout(60)  # a stalled connection raises instead of hanging forever\n\nBUCKET = "sen1floods11"\nPREFIXES = {\n    "S1Hand": "v1.1/data/flood_events/HandLabeled/S1Hand/",\n    "LabelHand": "v1.1/data/flood_events/HandLabeled/LabelHand/",\n    "flood_handlabeled": "v1.1/splits/flood_handlabeled/",\n}\n\n\ndef list_objects(prefix):\n    names, token = [], None\n    while True:\n        q = {"prefix": prefix, "maxResults": "1000", "fields": "items(name,size),nextPageToken"}\n        if token:\n            q["pageToken"] = token\n        url = f"https://storage.googleapis.com/storage/v1/b/{BUCKET}/o?" + urllib.parse.urlencode(q)\n        with urllib.request.urlopen(url, timeout=60) as r:\n            page = json.load(r)\n        names += [(it["name"], int(it["size"])) for it in page.get("items", []) if not it["name"].endswith("/")]\n        token = page.get("nextPageToken")\n        if not token:\n            return names\n\n\ndef fetch(name, size, dest):\n    if os.path.exists(dest) and os.path.getsize(dest) == size:\n        return "skip"\n    url = f"https://storage.googleapis.com/{BUCKET}/" + urllib.parse.quote(name)\n    for attempt in range(4):\n        try:\n            tmp = dest + ".part"\n            urllib.request.urlretrieve(url, tmp)\n            os.replace(tmp, dest)\n            return "ok"\n        except Exception as e:  # retry transient network errors\n            if attempt == 3:\n                return f"failed: {e}"\n            time.sleep(2 * (attempt + 1))\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--out", default="data")\n    ap.add_argument("--workers", type=int, default=8)\n    args = ap.parse_args()\n\n    jobs = []\n    for folder, prefix in PREFIXES.items():\n        os.makedirs(os.path.join(args.out, folder), exist_ok=True)\n        objs = list_objects(prefix)\n        print(f"{folder}: {len(objs)} files, {sum(s for _, s in objs) / 1e6:.0f} MB")\n        jobs += [(n, s, os.path.join(args.out, folder, os.path.basename(n))) for n, s in objs]\n\n    done, failed = 0, []\n    with ThreadPoolExecutor(args.workers) as ex:\n        for (n, _, _), res in zip(jobs, ex.map(lambda j: fetch(*j), jobs)):\n            done += 1\n            if res.startswith("failed"):\n                failed.append((n, res))\n            if done % 100 == 0 or done == len(jobs):\n                print(f"{done}/{len(jobs)} files")\n    if failed:\n        print(f"{len(failed)} files failed; run the script again to retry them:")\n        for n, r in failed[:10]:\n            print(" ", n, r)\n        sys.exit(1)\n    print("Download complete.")\n\n\nif __name__ == "__main__":\n    main()\n', 'dem_fetch.py': '"""Fetch the Copernicus GLO-30 DEM onto every Sen1Floods11 chip grid (data/DEM/<chip>_DEM.tif), as the PC cache did."""\nimport argparse\nimport math\nimport os\nfrom concurrent.futures import ThreadPoolExecutor\n\nos.environ.setdefault("GDAL_CACHEMAX", "64")\nos.environ.setdefault("CPL_VSIL_CURL_ALLOWED_EXTENSIONS", ".tif")\nos.environ.setdefault("GDAL_HTTP_TIMEOUT", "60")\nos.environ.setdefault("GDAL_HTTP_MAX_RETRY", "3")\n\nimport numpy as np\nimport rasterio\nfrom rasterio.warp import Resampling, reproject, transform_bounds\n\nDEM_URL = ("https://copernicus-dem-30m.s3.amazonaws.com/"\n           "Copernicus_DSM_COG_10_{ns}{lat:02d}_00_{ew}{lon:03d}_00_DEM/"\n           "Copernicus_DSM_COG_10_{ns}{lat:02d}_00_{ew}{lon:03d}_00_DEM.tif")\n\n\ndef fetch(s1_path, dst):\n    if os.path.exists(dst):\n        return 0\n    with rasterio.open(s1_path) as ref:\n        prof, b = ref.profile, ref.bounds\n        if ref.crs.is_projected:\n            b = transform_bounds(ref.crs, "EPSG:4326", *b)\n    dem = np.full((prof["height"], prof["width"]), np.nan, np.float32)\n    for lat in range(math.floor(b[1]), math.floor(b[3]) + 1):\n        for lon in range(math.floor(b[0]), math.floor(b[2]) + 1):\n            url = DEM_URL.format(ns="N" if lat >= 0 else "S", lat=abs(lat), ew="E" if lon >= 0 else "W", lon=abs(lon))\n            for t in range(3):\n                try:\n                    with rasterio.open("/vsicurl/" + url) as src:\n                        part = np.full_like(dem, np.nan)\n                        reproject(rasterio.band(src, 1), part, dst_transform=prof["transform"], dst_crs=prof["crs"],\n                                  dst_nodata=np.nan, resampling=Resampling.bilinear)\n                        dem = np.where(np.isnan(dem), part, dem)\n                    break\n                except Exception as e:\n                    if "404" in str(e):\n                        break  # no tile (sea)\n    if np.all(np.isnan(dem)):\n        return 0\n    dem = np.where(np.isnan(dem), np.nanmin(dem), dem)\n    prof.update(count=1, dtype="float32", nodata=None, compress="deflate")\n    os.makedirs(os.path.dirname(dst), exist_ok=True)\n    with rasterio.open(dst, "w", **prof) as d:\n        d.write(dem, 1)\n    return 1\n\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--data", default="data")\n    ap.add_argument("--workers", type=int, default=8)\n    a = ap.parse_args()\n    chips = sorted(f for f in os.listdir(os.path.join(a.data, "S1Hand")) if f.endswith(".tif"))\n    with ThreadPoolExecutor(a.workers) as ex:\n        n = sum(ex.map(lambda s: fetch(os.path.join(a.data, "S1Hand", s), os.path.join(a.data, "DEM", s.replace("S1Hand", "DEM"))), chips))\n    print(f"DEM: {n} new chips of {len(chips)}", flush=True)\n    print("DEM_DONE", flush=True)\n', 'hand_fetch.py': '"""\nFetch HAND (height above nearest drainage, GLO-30 HAND v1/2021, CC0, ASF/AWS) onto every chip grid.\n\n  Sen1Floods11: data/HAND/<chip>_HAND.tif on the S1 chip grid (same layout as data/DEM)\n  Kuro Siwo:    data_ks_hand/<split>/<actid>/<chip>.npy, float32 224x224, chip grid taken from the\n                footprint in data_ks/index.csv (EPSG:3857, 224 px of 10 m)\n\nHAND is the vertical distance (m) from a pixel to the nearest drainage cell it flows to: water\nponds near HAND 0, so it is the physically meaningful version of our "height above the chip\'s\nlow ground". Pixels without a tile (open sea) are set to 0.\n\nUsage:  python hand_fetch.py [--sen1floods11] [--kurosiwo] [--workers 8]\nPrints HAND_DONE at the end; resumable (existing outputs are skipped).\n"""\nimport argparse\nimport csv\nimport math\nimport os\nimport re\nimport socket\nimport time\nimport urllib.error\nimport urllib.request\nfrom concurrent.futures import ThreadPoolExecutor\n\nimport threading\n\nsocket.setdefaulttimeout(120)\n_locks, _locks_guard = {}, threading.Lock()\n_stats = {"chips_without_hand": 0}\nTILE_DIR = os.path.join("data", "HAND_tiles")\n\nos.environ.setdefault("GDAL_CACHEMAX", "64")\nos.environ.setdefault("CPL_VSIL_CURL_ALLOWED_EXTENSIONS", ".tif")\nos.environ.setdefault("GDAL_HTTP_TIMEOUT", "60")\nos.environ.setdefault("GDAL_HTTP_MAX_RETRY", "3")\nos.environ.setdefault("GDAL_DISABLE_READDIR_ON_OPEN", "EMPTY_DIR")\n\nimport numpy as np\nimport rasterio\nfrom rasterio.transform import from_bounds\nfrom rasterio.warp import Resampling, reproject, transform_bounds\n\nBASE = "https://glo-30-hand.s3.us-west-2.amazonaws.com/v1/2021/"\nNAME = "Copernicus_DSM_COG_10_{ns}{lat:02d}_00_{ew}{lon:03d}_00_HAND.tif"\nPATTERNS = [BASE + NAME, BASE + NAME.replace("_HAND.tif", "_HAND/") + NAME]  # flat, or one folder per tile\nNAME_ONLY = NAME\n_pattern = [None]\n\n\ndef tile_url(lat, lon, pattern):\n    return pattern.format(ns="N" if lat >= 0 else "S", lat=abs(lat), ew="E" if lon >= 0 else "W", lon=abs(lon))\n\n\ndef download_tile(lat, lon):\n    """Download one HAND tile in full (cached, one download per tile even with many threads)."""\n    name = tile_url(lat, lon, NAME)\n    with _locks_guard:\n        lock = _locks.setdefault(name, threading.Lock())\n    with lock:\n        return _download_tile(lat, lon, name)\n\n\ndef _download_tile(lat, lon, name):\n    dst = os.path.join(TILE_DIR, name)\n    if os.path.exists(dst):\n        return dst\n    if os.path.exists(dst + ".none"):\n        return None\n    os.makedirs(TILE_DIR, exist_ok=True)\n    url = tile_url(lat, lon, _pattern[0])\n    for t in range(4):\n        try:\n            urllib.request.urlretrieve(url, dst + ".part")\n            os.replace(dst + ".part", dst)\n            return dst\n        except urllib.error.HTTPError as e:\n            if e.code in (403, 404):  # S3 answers 403/404 for keys that do not exist (open sea)\n                open(dst + ".none", "w").close()\n                return None\n            err = e\n        except Exception as e:\n            err = e\n        time.sleep(5 * (t + 1))\n    print(f"  tile {name} failed: {type(err).__name__} {err}", flush=True)\n    return None\n\n\ndef probe():\n    """Find which URL layout the bucket uses, by downloading a land tile in Bangladesh (N23 E090)."""\n    for p in PATTERNS:\n        _pattern[0] = p\n        url = tile_url(23, 90, p)\n        try:\n            path = download_tile(23, 90)\n            if path is None:\n                raise IOError("missing")\n            with rasterio.open(path) as src:\n                v = src.read(1, out_shape=(1, 64, 64))\n            print(f"HAND layout ok: {url}  mean {float(np.nanmean(v)):.2f} m, size {os.path.getsize(path) / 2**20:.0f} MB", flush=True)\n            return True\n        except Exception as e:\n            print(f"  not {url}: {type(e).__name__} {str(e)[:200]}", flush=True)\n            for ext in ("", ".none", ".part"):\n                f = os.path.join(TILE_DIR, tile_url(23, 90, NAME)) + ext\n                if os.path.exists(f):\n                    os.remove(f)\n    return False\n\n\ndef hand_on_grid(transform, crs, shape):\n    left, top = transform.c, transform.f\n    right, bottom = left + transform.a * shape[1], top + transform.e * shape[0]\n    b = transform_bounds(crs, "EPSG:4326", left, bottom, right, top) if not crs.is_geographic else (left, bottom, right, top)\n    out = np.full(shape, np.nan, np.float32)\n    for lat in range(math.floor(b[1]), math.floor(b[3]) + 1):\n        for lon in range(math.floor(b[0]), math.floor(b[2]) + 1):\n            path = download_tile(lat, lon)\n            if path is None:\n                continue\n            try:\n                with rasterio.open(path) as src:\n                    part = np.full(shape, np.nan, np.float32)\n                    reproject(rasterio.band(src, 1), part, dst_transform=transform, dst_crs=crs,\n                              dst_nodata=np.nan, resampling=Resampling.bilinear)\n                    out = np.where(np.isnan(out), part, out)\n            except Exception as e:\n                print(f"  reading {path} failed: {e}", flush=True)\n                continue\n    if np.all(np.isnan(out)):\n        _stats["chips_without_hand"] += 1\n    return np.nan_to_num(out, nan=0.0)\n\n\ndef do_s1f(args):\n    from sen1floods11_unet_baseline import read_split\n    chips = [s for k in ["train", "valid", "test", "bolivia"] for s, _ in read_split(args.data, k)]\n    os.makedirs(os.path.join(args.data, "HAND"), exist_ok=True)\n\n    def one(s):\n        dst = os.path.join(args.data, "HAND", s.replace("S1Hand", "HAND"))\n        if os.path.exists(dst):\n            if not args.verify:\n                return 0\n            with rasterio.open(dst) as d:\n                if np.any(d.read(1) != 0):\n                    return 0\n            dem_p = os.path.join(args.data, "DEM", s.replace("S1Hand", "DEM"))\n            if os.path.exists(dem_p):\n                with rasterio.open(dem_p) as d:\n                    if float(np.nanstd(d.read(1))) < 2.0:\n                        return 0  # flat (sea or delta): an all-zero HAND is plausible\n            print(f"  re-fetching blank HAND for {s}", flush=True)\n        with rasterio.open(os.path.join(args.data, "S1Hand", s)) as ref:\n            prof, tr, crs, shape = ref.profile, ref.transform, ref.crs, (ref.height, ref.width)\n        h = hand_on_grid(tr, crs, shape)\n        prof.update(count=1, dtype="float32", nodata=None, compress="deflate")\n        with rasterio.open(dst + ".part.tif", "w", **prof) as d:\n            d.write(h, 1)\n        os.replace(dst + ".part.tif", dst)\n        return 1\n\n    with ThreadPoolExecutor(args.workers) as ex:\n        n = sum(ex.map(one, chips))\n    print(f"Sen1Floods11: {n} new HAND chips ({len(chips)} total)", flush=True)\n\n\ndef do_ks(args):\n    rows = list(csv.DictReader(open(os.path.join(args.ks, "index.csv"))))\n    crs = rasterio.crs.CRS.from_epsg(3857)\n\n    def one(r):\n        dst = os.path.join(args.ks_hand, r["file"].replace(".npz", ".npy"))\n        if os.path.exists(dst):\n            if not args.verify or np.any(np.load(dst) != 0):\n                return 0\n            z = np.load(os.path.join(args.ks, r["file"]))\n            if float(np.nanstd(z["dem"])) < 2.0:\n                return 0\n            print(f"  re-fetching blank HAND for {r[\'file\']}", flush=True)\n        nums = [float(v) for v in re.findall(r"-?\\d+\\.?\\d*", r["geom"])]\n        xs, ys = nums[0::2], nums[1::2]\n        tr = from_bounds(min(xs), min(ys), max(xs), max(ys), 224, 224)\n        h = hand_on_grid(tr, crs, (224, 224))\n        os.makedirs(os.path.dirname(dst), exist_ok=True)\n        np.save(dst + ".part.npy", h)\n        os.replace(dst + ".part.npy", dst)\n        return 1\n\n    with ThreadPoolExecutor(args.workers) as ex:\n        n = sum(ex.map(one, rows))\n    print(f"Kuro Siwo: {n} new HAND chips ({len(rows)} total)", flush=True)\n\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--data", default="data")\n    ap.add_argument("--ks", default="data_ks")\n    ap.add_argument("--ks-hand", default="data_ks_hand")\n    ap.add_argument("--sen1floods11", action="store_true")\n    ap.add_argument("--kurosiwo", action="store_true")\n    ap.add_argument("--workers", type=int, default=8)\n    ap.add_argument("--verify", action="store_true", help="re-fetch chips whose HAND is all zero over land with relief")\n    a = ap.parse_args()\n    if not (a.sen1floods11 or a.kurosiwo):\n        a.sen1floods11 = a.kurosiwo = True\n    if not probe():\n        raise SystemExit("HAND bucket not reachable with the expected file names")\n    if a.sen1floods11:\n        do_s1f(a)\n    if a.kurosiwo:\n        do_ks(a)\n    print(f"chips with no HAND coverage (all sea or failed): {_stats[\'chips_without_hand\']}", flush=True)\n    print("HAND_DONE", flush=True)\n', 'ks_download.py': '"""\nDownload a systematic subsample of the Kuro Siwo GRD webdataset (43 flood events) without\npulling the full 0.5+ TB.\n\nInside each tar shard, samples are stored event by event and are nearly fixed-size\n(about 1.74 MB), so we jump through every shard with HTTP range requests: every STEP_MB\nwe read a short window and keep the first K complete samples found there. This gives an\nevenly spread subsample of every event (roughly 1 in 50 patches), about 10 GB of traffic.\n\nKept per sample (data_ks/<split>/<actid>/<shard>_<offsetMB>_<key>.npz, compressed):\n  vv, vh            post-flood backscatter, dB, float16\n  vv_pre, vh_pre    first pre-flood image (sec1), dB, float16\n  dem               float32 metres\n  mask              uint8: 0 no water, 1 permanent water, 2 flood, 3 invalid\n  valid             uint8\nand info fields (actid, aoiid, flood_date, pflood, pwater, geom) in data_ks/index.csv.\nResumable: finished windows are listed in data_ks/progress.json.\n\nUsage:  python ks_download.py [--splits test_GRD train_GRD] [--step-mb 512] [--k 6]\n"""\nimport argparse\nimport csv\nimport io\nimport json\nimport os\nimport socket\nimport tarfile\nimport time\nimport urllib.request\n\nimport numpy as np\n\nsocket.setdefaulttimeout(60)\nREPO = "orion-ai-lab/Kuro-Siwo-Webdataset"\nBASE = f"https://huggingface.co/datasets/{REPO}/resolve/main"\nAPI = f"https://huggingface.co/api/datasets/{REPO}/tree/main"\nUA = {"User-Agent": "px1-research-download"}\nFIELDS = ["flood_vv", "flood_vh", "sec1_vv", "sec1_vh", "dem", "mask", "valid_mask", "info"]\n\n\ndef get(url, rng=None, tries=4):\n    for t in range(tries):\n        try:\n            h = dict(UA)\n            if rng:\n                h["Range"] = f"bytes={rng[0]}-{rng[1]}"\n            with urllib.request.urlopen(urllib.request.Request(url, headers=h)) as r:\n                return r.read()\n        except Exception as e:\n            if t == tries - 1:\n                raise\n            print(f"    retry {t + 1}: {type(e).__name__} {e}", flush=True)\n            time.sleep(5 * (t + 1))\n\n\ndef list_shards(split):\n    items = json.loads(get(f"{API}/{split}"))\n    return sorted((it["path"], it["size"]) for it in items if it.get("type") == "file" and it["path"].endswith(".tar"))\n\n\ndef first_header(buf):\n    for i in range(0, len(buf) - 512, 512):\n        blk = buf[i:i + 512]\n        if blk[257:262] != b"ustar":\n            continue\n        try:\n            tarfile.TarInfo.frombuf(blk, tarfile.ENCODING, "surrogateescape")\n            return i\n        except Exception:\n            continue\n    return None\n\n\ndef to_db(x):\n    return (10 * np.log10(np.clip(x, 1e-5, None))).astype(np.float16)\n\n\ndef parse_window(buf, k):\n    i = first_header(buf)\n    if i is None:\n        return []\n    out, cur, cur_key = [], {}, None\n    try:\n        tf = tarfile.open(fileobj=io.BytesIO(buf[i:]), mode="r|")\n        for m in tf:\n            if not m.isfile():\n                continue\n            key, _, field = m.name.partition(".")\n            field = field.replace(".npy", "").replace(".json", "")\n            if key != cur_key:\n                if cur_key is not None and all(f in cur for f in FIELDS):\n                    out.append((cur_key, cur))\n                    if len(out) >= k:\n                        return out\n                cur, cur_key = {}, key\n            if field in FIELDS:\n                cur[field] = tf.extractfile(m).read()\n    except (tarfile.ReadError, EOFError, OSError):\n        pass  # window ends mid-member\n    return out\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--splits", nargs="+", default=["test_GRD", "train_GRD"])\n    ap.add_argument("--step-mb", type=int, default=512)\n    ap.add_argument("--k", type=int, default=6)\n    ap.add_argument("--out", default="data_ks")\n    a = ap.parse_args()\n    os.makedirs(a.out, exist_ok=True)\n    prog_path = os.path.join(a.out, "progress.json")\n    done = set(json.load(open(prog_path))) if os.path.exists(prog_path) else set()\n    idx_path = os.path.join(a.out, "index.csv")\n    new_idx = not os.path.exists(idx_path)\n    idx_f = open(idx_path, "a", newline="")\n    idx = csv.writer(idx_f)\n    if new_idx:\n        idx.writerow(["split", "actid", "aoiid", "file", "flood_date", "pflood", "pwater", "water_px", "valid_px", "geom"])\n    win = (a.k + 1) * int(1.8 * 2**20) + 2**20\n    t0, total = time.time(), 0\n    for split in a.splits:\n        shards = list_shards(split)\n        print(f"{split}: {len(shards)} shards, {sum(s for _, s in shards) / 2**30:.0f} GB", flush=True)\n        for path, size in shards:\n            shard = os.path.basename(path)[:-4]\n            events = {}\n            for off in range(0, size, a.step_mb << 20):\n                tag = f"{split}/{shard}/{off >> 20}"\n                if tag in done:\n                    continue\n                buf = get(f"{BASE}/{path}", (off, min(size - 1, off + win)))\n                total += len(buf)\n                for key, s in parse_window(buf, a.k):\n                    info = json.loads(s["info"])\n                    act = str(info.get("actid"))\n                    arr = {f: np.load(io.BytesIO(s[f]))[0] for f in FIELDS if f != "info"}\n                    mask = arr["mask"].astype(np.uint8)\n                    d = os.path.join(a.out, split, act)\n                    os.makedirs(d, exist_ok=True)\n                    fn = f"{shard}_{off >> 20}_{key}.npz"\n                    np.savez_compressed(os.path.join(d, fn), vv=to_db(arr["flood_vv"]), vh=to_db(arr["flood_vh"]),\n                                        vv_pre=to_db(arr["sec1_vv"]), vh_pre=to_db(arr["sec1_vh"]),\n                                        dem=arr["dem"].astype(np.float32), mask=mask,\n                                        valid=arr["valid_mask"].astype(np.uint8))\n                    idx.writerow([split, act, info.get("aoiid"), f"{split}/{act}/{fn}", info.get("flood_date"),\n                                  info.get("pflood"), info.get("pwater"), int(((mask == 1) | (mask == 2)).sum()),\n                                  int((mask != 3).sum()), info.get("geom")])\n                    events[act] = events.get(act, 0) + 1\n                idx_f.flush()\n                done.add(tag)\n                with open(prog_path + ".tmp", "w") as f:\n                    json.dump(sorted(done), f)\n                os.replace(prog_path + ".tmp", prog_path)\n            el = time.time() - t0\n            print(f"  {shard}: events {events}  | {total / 2**30:.2f} GB in {el / 60:.0f} min "\n                  f"({total / 2**20 / max(el, 1):.1f} MB/s)", flush=True)\n    idx_f.close()\n    print("KS_DOWNLOAD_DONE", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'baselines_v2.py': '"""PX1 event-wise segmentation ports. See BASELINES_V2_PROTOCOL.md.\n\nIndependent implementation of mechanisms described by TENT, EATA, SAR and CoTTA.\nNo labels enter these adapters. Prediction never performs an optimiser update.\n"""\nfrom copy import deepcopy\nfrom dataclasses import dataclass, asdict\nimport math\n\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nMETHODS = (\'tent_v2\', \'eata_v2\', \'sar_v2\', \'cotta_v2\')\n\n\n@dataclass(frozen=True)\nclass Config:\n    lr: float = 1e-3\n    teacher_lr: float = 1e-4\n    entropy_margin: float = 0.4 * math.log(2)\n    diversity_margin: float = 0.05\n    fisher_weight: float = 2000.0\n    rho: float = 0.05\n    recovery_entropy: float = 0.2\n    ema_decay: float = 0.99\n    restore_probability: float = 0.01\n    anchor_confidence: float = 0.9\n    views: int = 32\n\n    def __post_init__(self):\n        for key, value in asdict(self).items():\n            if not math.isfinite(value):\n                raise ValueError(f\'Nonfinite configuration: {key}\')\n        if self.lr <= 0 or self.teacher_lr <= 0 or self.views < 1:\n            raise ValueError(\'Positive learning rates and view count required\')\n        if self.entropy_margin <= 0 or self.rho <= 0 or self.fisher_weight < 0:\n            raise ValueError(\'Invalid loss settings\')\n        for value in (self.diversity_margin, self.ema_decay, self.restore_probability, self.anchor_confidence):\n            if not 0 <= value <= 1:\n                raise ValueError(\'Probability/decay/margin must be in [0,1]\')\n\n\ndef entropy(logits):\n    return -(logits.softmax(1) * logits.log_softmax(1)).sum(1)\n\n\ndef configure(model, all_weights=False):\n    model.train()\n    model.requires_grad_(all_weights)\n    found = False\n    for module in model.modules():\n        if isinstance(module, nn.BatchNorm2d):\n            found = True\n            if not module.affine:\n                raise ValueError(\'Affine BatchNorm required\')\n            module.track_running_stats = False\n            module.running_mean = None\n            module.running_var = None\n            module.requires_grad_(True)\n    if not found:\n        raise ValueError(\'This version supports BatchNorm2d models only\')\n    return {name: p for name, p in model.named_parameters() if p.requires_grad}\n\n\ndef estimate_source_fisher(source_model, batches):\n    """Mean squared batch pseudo-label gradient on SOURCE inputs; no model mutation.\n\n    The caller supplies source-training crops, excluding the held-out source event.\n    Returns CPU tensors, reusable for the same checkpoint/normalisation/source split.\n    """\n    model = deepcopy(source_model)\n    parameters = configure(model)\n    result = {name: torch.zeros_like(p) for name, p in parameters.items()}\n    count = 0\n    for x in batches:\n        logits = model(x)\n        loss = F.cross_entropy(logits, logits.detach().argmax(1))\n        gradients = torch.autograd.grad(loss, tuple(parameters.values()))\n        for name, grad in zip(parameters, gradients):\n            if not torch.isfinite(grad).all():\n                raise FloatingPointError(\'Nonfinite source Fisher gradient\')\n            result[name].add_(grad.detach().square())\n        count += 1\n    if count == 0:\n        raise ValueError(\'EATA requires nonempty source Fisher batches\')\n    return {name: (value / count).cpu() for name, value in result.items()}\n\n\nclass Adapter:\n    def __init__(self, source_model, method, config=None, fisher=None):\n        if method not in METHODS:\n            raise ValueError(f\'Unknown baseline {method}\')\n        self.method, self.config = method, config or Config()\n        self.model = deepcopy(source_model)\n        self.parameters = configure(self.model, all_weights=method == \'cotta_v2\')\n        self.initial = deepcopy(self.model.state_dict())\n        lr = self.config.teacher_lr if method == \'cotta_v2\' else self.config.lr\n        self.optimizer = torch.optim.Adam(self.parameters.values(), lr=lr)\n        self.initial_optimizer = deepcopy(self.optimizer.state_dict())\n        self.fisher = {}\n        if method == \'eata_v2\':\n            if fisher is None or set(fisher) != set(self.parameters):\n                raise ValueError(\'EATA requires Fisher for every adapted source parameter\')\n            for name, parameter in self.parameters.items():\n                value = fisher[name].to(parameter)\n                if value.shape != parameter.shape or not torch.isfinite(value).all() or (value < 0).any():\n                    raise ValueError(f\'Invalid Fisher tensor: {name}\')\n                self.fisher[name] = value.detach().clone()\n        self.teacher = self.anchor = None\n        if method == \'cotta_v2\':\n            self.teacher = deepcopy(self.model).requires_grad_(False)\n            self.anchor = deepcopy(self.model).requires_grad_(False)\n        self.probability_average = None\n        self.entropy_average = None\n        self.stats = dict(steps=0, updates=0, skipped=0, reliable_pixels=0,\n                          selected_pixels=0, recoveries=0, augmented_calls=0, plain_calls=0)\n\n    def reset(self):\n        self.model.load_state_dict(self.initial)\n        self.optimizer.load_state_dict(deepcopy(self.initial_optimizer))\n        self.optimizer.zero_grad(set_to_none=True)\n        self.probability_average = self.entropy_average = None\n        if self.teacher is not None:\n            self.teacher.load_state_dict(self.initial)\n            self.anchor.load_state_dict(self.initial)\n\n    def _update(self, loss):\n        if not torch.isfinite(loss):\n            raise FloatingPointError(\'Nonfinite adaptation loss\')\n        self.optimizer.zero_grad(set_to_none=True)\n        loss.backward()\n        if any(p.grad is not None and not torch.isfinite(p.grad).all() for p in self.parameters.values()):\n            self.optimizer.zero_grad(set_to_none=True)\n            raise FloatingPointError(\'Nonfinite adaptation gradient\')\n        self.optimizer.step()\n        self.stats[\'updates\'] += 1\n\n    def _eata(self, x):\n        logits = self.model(x)\n        h = entropy(logits)\n        reliable = h < self.config.entropy_margin\n        probabilities = logits.softmax(1).movedim(1, -1)[reliable]\n        losses = h[reliable]\n        self.stats[\'reliable_pixels\'] += losses.numel()\n        if probabilities.numel() and self.probability_average is not None:\n            similarity = F.cosine_similarity(probabilities.detach(), self.probability_average[None], dim=1)\n            diverse = similarity.abs() < self.config.diversity_margin\n            probabilities, losses = probabilities[diverse], losses[diverse]\n        self.stats[\'selected_pixels\'] += losses.numel()\n        if losses.numel() == 0:\n            self.stats[\'skipped\'] += 1\n            return\n        current = probabilities.detach().mean(0)\n        self.probability_average = current if self.probability_average is None else 0.9 * self.probability_average + 0.1 * current\n        weighted = (losses * torch.exp(self.config.entropy_margin - losses.detach())).mean()\n        penalty = sum((self.fisher[n] * (p - self.initial[n]).square()).sum() for n, p in self.parameters.items())\n        self._update(weighted + self.config.fisher_weight * penalty)\n\n    def _sar(self, x):\n        h = entropy(self.model(x))\n        selected = h < self.config.entropy_margin\n        self.stats[\'reliable_pixels\'] += int(selected.sum())\n        if not selected.any():\n            self.stats[\'skipped\'] += 1\n            return\n        parameters = tuple(self.parameters.values())\n        gradients = torch.autograd.grad(h[selected].mean(), parameters)\n        norm = torch.stack([g.square().sum() for g in gradients]).sum().sqrt()\n        if not torch.isfinite(norm):\n            raise FloatingPointError(\'Nonfinite sharpness gradient\')\n        originals = [p.detach().clone() for p in parameters]\n        self.optimizer.zero_grad(set_to_none=True)\n        value = None\n        try:\n            with torch.no_grad():\n                for p, grad in zip(parameters, gradients):\n                    p.add_(grad * (self.config.rho / norm.clamp_min(1e-12)))\n            h2 = entropy(self.model(x))\n            keep = selected & (h2 < self.config.entropy_margin)\n            self.stats[\'selected_pixels\'] += int(keep.sum())\n            if keep.any():\n                loss = h2[keep].mean()\n                if not torch.isfinite(loss):\n                    raise FloatingPointError(\'Nonfinite sharpness loss\')\n                loss.backward()\n                value = float(loss.detach())\n        finally:\n            # Restore exactly even if the second filter is empty or forward raises.\n            with torch.no_grad():\n                for p, original in zip(parameters, originals):\n                    p.copy_(original)\n        if value is None:\n            self.stats[\'skipped\'] += 1\n            self.optimizer.zero_grad(set_to_none=True)\n            return\n        if any(p.grad is not None and not torch.isfinite(p.grad).all() for p in parameters):\n            self.optimizer.zero_grad(set_to_none=True)\n            raise FloatingPointError(\'Nonfinite second sharpness gradient\')\n        self.optimizer.step()\n        self.stats[\'updates\'] += 1\n        self.entropy_average = value if self.entropy_average is None else 0.9 * self.entropy_average + 0.1 * value\n        if self.entropy_average < self.config.recovery_entropy:\n            self.reset()\n            self.stats[\'recoveries\'] += 1\n\n    @torch.no_grad()\n    def _teacher_prediction(self, x, augment):\n        confidence = self.anchor(x).softmax(1).amax(1).mean()\n        if confidence >= self.config.anchor_confidence:\n            self.stats[\'plain_calls\'] += 1\n            return self.teacher(x)\n        if augment is None:\n            raise ValueError(\'CoTTA low-confidence prediction requires spatially invertible SAR augmentation\')\n        total = None\n        for _ in range(self.config.views):\n            view, undo = augment(x)\n            logits = undo(self.teacher(view))\n            total = logits if total is None else total + logits\n        self.stats[\'augmented_calls\'] += 1\n        return total / self.config.views\n\n    def _cotta(self, x, augment):\n        target = self._teacher_prediction(x, augment).softmax(1)\n        self._update(-(target * self.model(x).log_softmax(1)).sum(1).mean())\n        with torch.no_grad():\n            for teacher, student in zip(self.teacher.parameters(), self.model.parameters()):\n                teacher.lerp_(student, 1 - self.config.ema_decay)\n            for name, p in self.parameters.items():\n                restore = torch.rand_like(p) < self.config.restore_probability\n                p.copy_(torch.where(restore, self.initial[name], p))\n\n    def step(self, x, augment=None):\n        if not torch.isfinite(x).all():\n            raise ValueError(\'Nonfinite target input\')\n        self.stats[\'steps\'] += 1\n        self.model.train()\n        with torch.enable_grad():\n            if self.method == \'tent_v2\':\n                self._update(entropy(self.model(x)).mean())\n            elif self.method == \'eata_v2\':\n                self._eata(x)\n            elif self.method == \'sar_v2\':\n                self._sar(x)\n            else:\n                self._cotta(x, augment)\n\n    @torch.no_grad()\n    def predict(self, x, augment=None):\n        self.model.eval()\n        if self.teacher is not None:\n            self.teacher.eval()\n            self.anchor.eval()\n            logits = self._teacher_prediction(x, augment)\n        else:\n            logits = self.model(x)\n        if not torch.isfinite(logits).all():\n            raise FloatingPointError(\'Nonfinite prediction\')\n        return logits\n\n\ndef sar_augment(std):\n    """D4 geometry with exact inverse; +/-1.5 dB shared radiometric offset.\n\n    Operates on normalised VV/VH only; radiometric offsets are scaled by source std.\n    """\n    def transform(x):\n        if x.shape[1] != 2:\n            raise ValueError(\'SAR augmentation requires VV/VH input\')\n        k = int(torch.randint(4, ()).item())\n        flip = bool(torch.randint(2, ()).item())\n        value = torch.rot90(x, k, (-2, -1))\n        if flip:\n            value = value.flip(-1)\n        offset = (torch.rand((x.shape[0], 1, 1, 1), device=x.device) * 3 - 1.5)\n        value = value + offset / std.to(x).reshape(1, 2, 1, 1)\n        def inverse(logits):\n            if flip:\n                logits = logits.flip(-1)\n            return torch.rot90(logits, -k, (-2, -1))\n        return value, inverse\n    return transform\n', 'eval_baselines_v2.py': '"""Isolated event-wise evaluation on existing S1F/KS/MM data only.\n\nOutputs never match legacy histogram globs. No data downloads are performed here.\n"""\nimport argparse\nfrom dataclasses import asdict, replace\nimport hashlib\nimport json\nimport re\nfrom pathlib import Path\nimport sys\nimport time\n\nimport numpy as np\nimport torch\nfrom baselines_v2 import Adapter, Config, METHODS, estimate_source_fisher, sar_augment\n\nHERE = Path(__file__).resolve().parent\nif (HERE.parent / \'src\').exists():\n    sys.path.insert(0, str(HERE.parent / \'src\'))\nBINS = 1000\nSENSITIVITIES = {\'eata_v2_d09\': \'eata_v2\', \'sar_v2_scaled\': \'sar_v2\'}\nALL_METHODS = (\'none\', \'adabn\', \'adabn_physa\') + METHODS + tuple(SENSITIVITIES)\n\n\ndef method_config(method, config):\n    if method == \'eata_v2_d09\':\n        return replace(config, diversity_margin=0.9)\n    if method == \'sar_v2_scaled\':\n        return replace(config, recovery_entropy=0.2 * np.log(2) / np.log(1000))\n    return config\n\n\ndef digest(path):\n    h = hashlib.sha256()\n    with open(path, \'rb\') as f:\n        for chunk in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef write_json(path, value):\n    path.write_text(json.dumps(value, indent=2, allow_nan=False), encoding=\'utf-8\')\n\n\ndef source_pairs(pairs, held_out=None):\n    return [p for p in pairs if held_out is None or p[0].split(\'_\')[0] != held_out]\n\n\nclass Crops:\n    def __init__(self, raw, mean, std, device, seed=0, batch=8, size=256):\n        if not raw or any(r.shape[0] != 2 or not np.isfinite(r).all() for r in raw):\n            raise ValueError(\'Empty/invalid VV/VH inputs\')\n        self.raw, self.device = raw, device\n        self.mean = np.asarray(mean, dtype=\'float32\')[:, None, None]\n        self.std = np.asarray(std, dtype=\'float32\')[:, None, None]\n        if not np.isfinite(self.mean).all() or not np.isfinite(self.std).all() or (self.std <= 0).any():\n            raise ValueError(\'Invalid source normalisation\')\n        self.rng, self.batch = np.random.default_rng(seed), batch\n        self.size = min(size, *(min(r.shape[-2:]) for r in raw))\n\n    def __call__(self):\n        values = []\n        for index in self.rng.integers(0, len(self.raw), self.batch):\n            r = self.raw[index]\n            h, w = r.shape[-2:]\n            y, x = self.rng.integers(0, h - self.size + 1), self.rng.integers(0, w - self.size + 1)\n            value = (r[:, y:y+self.size, x:x+self.size] - self.mean) / self.std\n            value = np.rot90(value, int(self.rng.integers(4)), axes=(1, 2))\n            if self.rng.random() < 0.5:\n                value = value[:, :, ::-1]\n            values.append(value.copy())\n        return torch.from_numpy(np.stack(values)).to(self.device)\n\n    def image(self, raw):\n        return torch.from_numpy(((raw - self.mean) / self.std).copy())[None].to(self.device)\n\n\ndef score_arrays(model, method, ev, mean, std, config, fisher, steps, device, seed=0):\n    # Only raw inputs reach Adapter. Labels are consulted after adaptation completes.\n    torch.manual_seed(seed)\n    crops = Crops(ev[\'raw\'], mean, std, device, seed)\n    adapter = Adapter(model, method, config, fisher)\n    augment = sar_augment(torch.as_tensor(std))\n    for _ in range(steps):\n        adapter.step(crops(), augment)\n    hist = np.zeros((2, 2, BINS), dtype=np.int64)\n    with torch.no_grad():\n        for i, raw in enumerate(ev[\'raw\']):\n            p = adapter.predict(crops.image(raw), augment).softmax(1)[0, 1].cpu().numpy()\n            bins = np.clip((p * BINS).astype(\'int64\'), 0, BINS-1)\n            for subset, labels in [(0, ev[\'lab\'][i])] + ([(1, ev[\'lab_alt\'][i])] if \'lab_alt\' in ev else []):\n                if labels.shape != p.shape:\n                    raise ValueError(\'Prediction/label shape mismatch\')\n                for label in (0, 1):\n                    hist[subset, label] += np.bincount(bins[labels == label], minlength=BINS)\n    return hist, dict(adapter.stats)\n\n\ndef dataset_events(name, work):\n    from sen1floods11_unet_baseline import read_split, event_of\n    if name == \'ks\':\n        from ks_eval import events_of\n        return events_of(str(work / \'data_ks\'))\n    if name == \'mm\':\n        from mm_eval import events_of\n        return events_of(str(work / \'data_mm\'))\n    all_pairs = sum([read_split(str(work / \'data\'), s) for s in (\'train\', \'valid\', \'test\', \'bolivia\')], [])\n    return {event: [p for p in all_pairs if event_of(p[0]) == event] for event in sorted({event_of(p[0]) for p in all_pairs})}\n\n\ndef load_event(name, event, files, work):\n    if name == \'ks\':\n        from ks_eval import load_ks_event\n        ev = load_ks_event(files)\n        ev[\'lab_alt\'] = []\n        for file, label in zip(files, ev[\'lab\']):\n            with np.load(file) as data:\n                alt = label.copy()\n                alt[data[\'mask\'] == 1] = 255\n                ev[\'lab_alt\'].append(alt)\n        return ev\n    if name == \'mm\':\n        from mm_eval import load_mm_event\n        return load_mm_event(files)\n    from tta_px1 import load_event as load_s1f\n    args = argparse.Namespace(data=str(work / \'data\'))\n    return load_s1f(args, event, files, set())\n\n\ndef validate_histogram(hist, ev):\n    if hist.shape != (2, 2, BINS) or (hist < 0).any():\n        raise ValueError(\'Invalid histogram\')\n    for subset, key in [(0, \'lab\'), (1, \'lab_alt\')]:\n        expected = sum(int(((y == 0) | (y == 1)).sum()) for y in ev.get(key, []))\n        if int(hist[subset].sum()) != expected:\n            raise ValueError(f\'Histogram pixel count mismatch for {key}\')\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument(\'--dataset\', required=True, choices=[\'ks\', \'s1f\', \'mm\'])\n    p.add_argument(\'--work\', type=Path, default=Path(\'/tmp/px1\'))\n    p.add_argument(\'--out\', type=Path, required=True)\n    p.add_argument(\'--seed\', type=int, default=0)\n    p.add_argument(\'--methods\', default=\',\'.join(ALL_METHODS))\n    p.add_argument(\'--smoke\', action=\'store_true\')\n    p.add_argument(\'--device\', choices=[\'cuda\', \'cpu\'], default=\'cuda\')\n    args = p.parse_args()\n    if args.seed not in (0, 1, 2) or (args.dataset == \'s1f\' and args.seed != 0):\n        p.error(\'Existing checkpoints: external seeds 0/1/2; S1F LOEO seed 0\')\n    methods = args.methods.split(\',\')\n    if not methods or len(set(methods)) != len(methods) or set(methods) - set(ALL_METHODS):\n        p.error(\'Unknown or duplicate method\')\n    if args.device == \'cuda\' and not torch.cuda.is_available():\n        raise RuntimeError(\'CUDA required for this run; CPU unit tests are separate\')\n    work = args.work.resolve()\n    events = dataset_events(args.dataset, work)\n    if not events:\n        raise ValueError(\'No eligible events\')\n    if not args.smoke:\n        expected = {\'ks\': 26, \'s1f\': 11, \'mm\': 69}[args.dataset]\n        if len(events) != expected:\n            raise ValueError(f\'Unexpected event/region count {len(events)} != {expected}; inspect dataset before running\')\n        if args.dataset == \'mm\' and len({e.split(\'-\')[0] for e in events}) != 30:\n            raise ValueError(\'Expected 30 MMFlood activations\')\n    if args.smoke:\n        event = sorted(events)[0]\n        events = {event: events[event][:2]}\n    # Refuse reuse: completed and partial runs both remain immutable.\n    args.out.mkdir(parents=True, exist_ok=False)\n    config = Config()  # smoke uses the same 32-view CoTTA predictor as the full run\n    steps, fisher_batches = (2, 2) if args.smoke else (200, 20)\n    manifest = dict(schema=\'px1-baselines-v2\', dataset=args.dataset, seed=args.seed,\n        mode=\'smoke\' if args.smoke else \'full\', config=asdict(config), steps=steps,\n        fisher_batches=fisher_batches, methods=methods,\n        method_configs={m: asdict(method_config(m, config)) for m in methods if m in METHODS or m in SENSITIVITIES},\n        events=sorted(events), status=\'started\',\n        software=dict(python=sys.version, numpy=np.__version__, torch=torch.__version__, cuda=torch.version.cuda),\n        code={path.name: digest(path) for path in HERE.glob(\'*.py\')}, records=[])\n    write_json(args.out / \'manifest.json\', manifest)\n    try:\n        from sen1floods11_unet_baseline import UNet, read_split, load_s1\n        from tta_px1 import run_one\n        training = read_split(str(work / \'data\'), \'train\')\n        source_raw = {s: load_s1(str(work / \'data/S1Hand\' / s)) for s, _ in training}\n        source_hash = {s: digest(work / \'data/S1Hand\' / s) for s in source_raw}\n        cache = {}\n        for event, files in sorted(events.items()):\n            selected = source_pairs(training, event if args.dataset == \'s1f\' else None)\n            if not selected:\n                raise ValueError(\'Empty source training subset\')\n            checkpoint = work / (\'runs_loeo\' if args.dataset == \'s1f\' else \'runs\') / (\n                f\'V0_hold{event}_seed0\' if args.dataset == \'s1f\' else f\'V0_seed{args.seed}\') / \'model.pt\'\n            key = str(checkpoint)\n            if key not in cache:\n                values = [source_raw[s] for s, _ in selected]\n                joined = np.concatenate([r.reshape(2, -1) for r in values], axis=1)\n                mean, std = joined.mean(1).astype(\'float32\'), joined.std(1).astype(\'float32\') + 1e-6\n                model = UNet(cin=2).to(args.device)\n                model.load_state_dict(torch.load(checkpoint, map_location=args.device, weights_only=True))\n                fisher = None\n                if any(m.startswith(\'eata_v2\') for m in methods):\n                    torch.manual_seed(1729)\n                    source_crops = Crops(values, mean, std, args.device, seed=1729)\n                    fisher = estimate_source_fisher(model, (source_crops() for _ in range(fisher_batches)))\n                cache[key] = (model, mean, std, fisher)\n            model, mean, std, fisher = cache[key]\n            ev = load_event(args.dataset, event, files, work)\n            # Record actual input files, including label files for the S1F reader.\n            input_paths = ([work / \'data/S1Hand\' / s for s, _ in files] + [work / \'data/LabelHand\' / l for _, l in files]\n                           if args.dataset == \'s1f\' else [Path(f) for f in files])\n            record = dict(event=event, activation=event.split(\'-\')[0] if args.dataset == \'mm\' else event,\n                checkpoint_sha256=digest(checkpoint), source_images={s: source_hash[s] for s, _ in selected},\n                input_files={str(f): digest(f) for f in input_paths}, metrics={}, outputs={})\n            for method in methods:\n                start = time.monotonic()\n                if method in METHODS or method in SENSITIVITIES:\n                    hist, stats = score_arrays(model, SENSITIVITIES.get(method, method), ev, mean, std,\n                                              method_config(method, config), fisher, steps, args.device)\n                else:\n                    legacy_args = argparse.Namespace(seed=0, model_seed=args.seed, steps=steps, runs=str(work / \'runs\'))\n                    hist = run_one(method, \'V0\', event, ev, mean, std, legacy_args, args.device, ckpt=str(checkpoint))\n                    stats = {\'comparison_control\': True}\n                validate_histogram(hist, ev)\n                safe = re.sub(r\'[^A-Za-z0-9_-]\', \'_\', event)\n                output = args.out / f\'{method}__{safe}.npz\'\n                if output.exists():\n                    raise FileExistsError(output)\n                np.savez_compressed(output, histogram=hist)\n                record[\'outputs\'][method] = dict(file=output.name, sha256=digest(output))\n                record[\'metrics\'][method] = dict(stats, seconds=time.monotonic() - start)\n                print(event, method, record[\'metrics\'][method], flush=True)\n            manifest[\'records\'].append(record)\n            write_json(args.out / \'manifest.json\', manifest)\n            if args.dataset == \'s1f\':\n                cache.clear()  # never retain 11 GPU models\n        manifest[\'status\'] = \'completed\'\n    except BaseException as exc:\n        manifest[\'status\'], manifest[\'error\'] = \'failed\', repr(exc)\n        raise\n    finally:\n        write_json(args.out / \'manifest.json\', manifest)\n    print(\'BASELINES_V2_DONE\', flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'test_baselines_v2.py': '"""Mechanism regression tests. CPU only, entirely synthetic inputs."""\nimport os\nos.environ[\'CUDA_VISIBLE_DEVICES\'] = \'\'\nfrom copy import deepcopy\nfrom dataclasses import replace\nimport unittest\nfrom unittest.mock import patch\nimport torch\nfrom torch import nn\nfrom baselines_v2 import Adapter, Config, METHODS, configure, entropy, estimate_source_fisher, sar_augment\n\n\nclass SmallNet(nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.feature = nn.Conv2d(2, 4, 1)\n        self.bn = nn.BatchNorm2d(4)\n        self.head = nn.Conv2d(4, 2, 1)\n\n    def forward(self, x):\n        return self.head(self.bn(self.feature(x)))\n\n\nclass BaselineTests(unittest.TestCase):\n    def setUp(self):\n        torch.manual_seed(2026)\n        self.model = SmallNet()\n        self.x = torch.randn(2, 2, 8, 8)\n        self.fisher = estimate_source_fisher(self.model, [self.x, self.x * 0.7])\n\n    def assert_state_equal(self, a, b):\n        self.assertEqual(set(a), set(b))\n        for name in a:\n            torch.testing.assert_close(a[name], b[name], rtol=0, atol=0)\n\n    def test_all_baselines_finite_and_source_unchanged(self):\n        original = deepcopy(self.model.state_dict())\n        for method in METHODS:\n            adapter = Adapter(self.model, method, Config(views=2), self.fisher if method == \'eata_v2\' else None)\n            aug = sar_augment(torch.ones(2))\n            adapter.step(self.x, aug)\n            probability = adapter.predict(self.x, aug).softmax(1)\n            self.assertTrue(torch.isfinite(probability).all())\n            self.assert_state_equal(original, self.model.state_dict())\n            self.assertIsNone(adapter.model.bn.running_mean)\n            self.assertFalse(adapter.model.bn.track_running_stats)\n\n    def test_tent_matches_one_manual_entropy_step(self):\n        oracle = deepcopy(self.model)\n        params = configure(oracle)\n        optimizer = torch.optim.Adam(params.values(), lr=Config().lr)\n        logits = oracle(self.x)\n        loss = -(logits.softmax(1) * logits.log_softmax(1)).sum(1).mean()\n        loss.backward()\n        optimizer.step()\n        adapter = Adapter(self.model, \'tent_v2\')\n        adapter.step(self.x)\n        self.assert_state_equal(oracle.state_dict(), adapter.model.state_dict())\n\n    def test_fisher_source_only_nonnegative_and_no_mutation(self):\n        original = deepcopy(self.model.state_dict())\n        f = estimate_source_fisher(self.model, [self.x])\n        self.assertTrue(all((v >= 0).all() and torch.isfinite(v).all() for v in f.values()))\n        self.assertTrue(any((v > 0).any() for v in f.values()))\n        self.assert_state_equal(original, self.model.state_dict())\n        with self.assertRaises(ValueError):\n            estimate_source_fisher(self.model, [])\n        with self.assertRaises(ValueError):\n            Adapter(self.model, \'eata_v2\')\n\n    def test_eata_rejects_redundant_pixels_without_optimizer_step(self):\n        config = Config(entropy_margin=1.0, diversity_margin=0.0)\n        adapter = Adapter(self.model, \'eata_v2\', config, self.fisher)\n        adapter.step(self.x)\n        self.assertEqual(adapter.stats[\'updates\'], 1)\n        before = deepcopy(adapter.model.state_dict())\n        adapter.step(self.x)\n        self.assertEqual(adapter.stats[\'updates\'], 1)\n        self.assertEqual(adapter.stats[\'skipped\'], 1)\n        self.assert_state_equal(before, adapter.model.state_dict())\n\n    def test_eata_fisher_penalty_and_weighted_entropy(self):\n        config = Config(entropy_margin=1, diversity_margin=1)\n        adapter = Adapter(self.model, \'eata_v2\', config, self.fisher)\n        with torch.no_grad():\n            for p in adapter.parameters.values():\n                p.add_(0.1)\n        h = entropy(adapter.model(self.x))\n        expected_entropy = (h * torch.exp(config.entropy_margin - h.detach())).mean()\n        expected_penalty = sum((self.fisher[n] * (p - adapter.initial[n]).square()).sum()\n                               for n, p in adapter.parameters.items()) * config.fisher_weight\n        self.assertGreater(float(expected_penalty.detach()), 0)\n        with patch.object(adapter, \'_update\') as update:\n            adapter.step(self.x)\n        torch.testing.assert_close(update.call_args.args[0], expected_entropy + expected_penalty)\n\n    def test_only_bn_parameters_change_for_entropy_methods(self):\n        for method in (\'tent_v2\', \'eata_v2\', \'sar_v2\'):\n            adapter = Adapter(self.model, method, Config(entropy_margin=1, recovery_entropy=0),\n                              self.fisher if method == \'eata_v2\' else None)\n            before = deepcopy(adapter.model.state_dict())\n            adapter.step(self.x)\n            for name, value in adapter.model.state_dict().items():\n                if not name.startswith(\'bn.\'):\n                    torch.testing.assert_close(value, before[name], rtol=0, atol=0)\n\n    def test_source_holdout_filter_and_named_sensitivities(self):\n        from eval_baselines_v2 import source_pairs, method_config\n        pairs = [(\'Bolivia_1.tif\', \'unused\'), (\'India_1.tif\', \'unused\')]\n        self.assertEqual(source_pairs(pairs, \'Bolivia\'), [pairs[1]])\n        self.assertEqual(source_pairs(pairs), pairs)\n        self.assertEqual(method_config(\'eata_v2_d09\', Config()).diversity_margin, 0.9)\n        self.assertLess(method_config(\'sar_v2_scaled\', Config()).recovery_entropy, Config().recovery_entropy)\n\n    def test_empty_reliability_filter_is_finite_and_does_not_update(self):\n        for method in (\'eata_v2\', \'sar_v2\'):\n            adapter = Adapter(self.model, method, Config(entropy_margin=1e-12), self.fisher if method == \'eata_v2\' else None)\n            before = deepcopy(adapter.model.state_dict())\n            adapter.step(self.x)\n            self.assertEqual(adapter.stats[\'updates\'], 0)\n            self.assertEqual(adapter.stats[\'skipped\'], 1)\n            self.assert_state_equal(before, adapter.model.state_dict())\n\n    def test_sar_recovers_model_optimizer_and_entropy_state(self):\n        adapter = Adapter(self.model, \'sar_v2\', Config(entropy_margin=1, recovery_entropy=1))\n        before = deepcopy(adapter.model.state_dict())\n        adapter.step(self.x)\n        self.assertEqual(adapter.stats[\'recoveries\'], 1)\n        self.assertIsNone(adapter.entropy_average)\n        self.assertEqual(adapter.optimizer.state_dict()[\'state\'], {})\n        self.assert_state_equal(before, adapter.model.state_dict())\n\n    def test_sar_restores_perturbation_when_second_filter_empty(self):\n        adapter = Adapter(self.model, \'sar_v2\', Config(entropy_margin=1))\n        before = deepcopy(adapter.model.state_dict())\n        count = 0\n        def changing_entropy(logits):\n            nonlocal count\n            count += 1\n            h = entropy(logits)\n            return h if count == 1 else h + 10\n        with patch(\'baselines_v2.entropy\', side_effect=changing_entropy):\n            adapter.step(self.x)\n        self.assertEqual(adapter.stats[\'skipped\'], 1)\n        self.assert_state_equal(before, adapter.model.state_dict())\n\n    def test_sar_restores_perturbation_on_exception(self):\n        adapter = Adapter(self.model, \'sar_v2\', Config(entropy_margin=1))\n        before = deepcopy(adapter.model.state_dict())\n        with patch(\'baselines_v2.entropy\', side_effect=[entropy(adapter.model(self.x)), RuntimeError(\'test\')]):\n            with self.assertRaises(RuntimeError):\n                adapter.step(self.x)\n        self.assert_state_equal(before, adapter.model.state_dict())\n\n    def test_cotta_confidence_gate_and_logit_average(self):\n        for threshold, expected in [(0.0, 0), (1.0, 3)]:\n            adapter = Adapter(self.model, \'cotta_v2\', Config(anchor_confidence=threshold, views=3))\n            calls = []\n            def aug(x):\n                calls.append(1)\n                return x.flip(-1), lambda y: y.flip(-1)\n            output = adapter.predict(self.x, aug)\n            self.assertEqual(len(calls), expected)\n            torch.testing.assert_close(output, adapter.teacher(self.x), atol=1e-6, rtol=1e-6)\n\n    def test_cotta_restore_and_teacher_update(self):\n        adapter = Adapter(self.model, \'cotta_v2\', Config(anchor_confidence=0, restore_probability=1))\n        before = deepcopy(adapter.model.state_dict())\n        adapter.step(self.x)\n        self.assert_state_equal(before, adapter.model.state_dict())\n        self.assertTrue(any(not torch.equal(before[k], v) for k, v in adapter.teacher.state_dict().items() if v.is_floating_point()))\n        self.assert_state_equal(before, adapter.anchor.state_dict())\n        adapter.reset()\n        self.assert_state_equal(before, adapter.teacher.state_dict())\n\n    def test_predict_does_not_update_weights(self):\n        for method in METHODS:\n            adapter = Adapter(self.model, method, Config(views=2), self.fisher if method == \'eata_v2\' else None)\n            before = deepcopy(adapter.model.state_dict())\n            adapter.predict(self.x, sar_augment(torch.ones(2)))\n            self.assertEqual(adapter.stats[\'updates\'], 0)\n            self.assert_state_equal(before, adapter.model.state_dict())\n\n    def test_sar_augmentation_inverse_preserves_spatial_pattern(self):\n        aug = sar_augment(torch.tensor([2., 4.]))\n        x = torch.arange(2 * 6 * 8, dtype=torch.float32).reshape(1, 2, 6, 8)\n        for _ in range(8):\n            view, undo = aug(x)\n            difference = undo(view) - x\n            torch.testing.assert_close(difference, difference[:, :, :1, :1].expand_as(x), atol=1e-5, rtol=0)\n            torch.testing.assert_close(difference[:, 0] * 2, difference[:, 1] * 4, atol=2e-5, rtol=0)\n\n\nif __name__ == \'__main__\':\n    torch.set_num_threads(1)\n    unittest.main(verbosity=2)\n', 'verify_baseline_outputs.py': '"""Validate v2 manifests/histograms before downstream statistical analysis."""\nimport argparse\nimport json\nfrom pathlib import Path\nimport numpy as np\nfrom eval_baselines_v2 import digest, BINS\n\n\ndef verify(directory, allow_smoke=False):\n    directory = Path(directory)\n    manifest = json.loads((directory / \'manifest.json\').read_text())\n    if manifest.get(\'schema\') != \'px1-baselines-v2\' or manifest.get(\'status\') != \'completed\':\n        raise ValueError(\'Only completed v2 workers can be analysed\')\n    if manifest[\'mode\'] != \'full\' and not allow_smoke:\n        raise ValueError(\'Smoke results must not enter final analysis\')\n    events = manifest[\'events\']\n    records = manifest[\'records\']\n    if len(events) != len(set(events)) or sorted(r[\'event\'] for r in records) != sorted(events):\n        raise ValueError(\'Incomplete or duplicate event records\')\n    outputs = set()\n    for record in records:\n        if set(record[\'outputs\']) != set(manifest[\'methods\']):\n            raise ValueError(\'Incomplete method coverage\')\n        for item in record[\'outputs\'].values():\n            name = item[\'file\']\n            if Path(name).name != name or name in outputs:\n                raise ValueError(\'Invalid or duplicate output filename\')\n            outputs.add(name)\n            file = directory / name\n            if digest(file) != item[\'sha256\']:\n                raise ValueError(f\'Changed result file: {name}\')\n            with np.load(file, allow_pickle=False) as data:\n                hist = data[\'histogram\']\n                if hist.shape != (2, 2, BINS) or not np.issubdtype(hist.dtype, np.integer) or (hist < 0).any():\n                    raise ValueError(f\'Invalid histogram: {name}\')\n    if outputs != {p.name for p in directory.glob(\'*.npz\')}:\n        raise ValueError(\'Unrecorded histogram files present\')\n    return dict(dataset=manifest[\'dataset\'], seed=manifest[\'seed\'], mode=manifest[\'mode\'],\n                events=len(events), methods=manifest[\'methods\'], histogram_files=len(outputs), status=\'verified\')\n\n\nif __name__ == \'__main__\':\n    p = argparse.ArgumentParser()\n    p.add_argument(\'directory\', type=Path)\n    p.add_argument(\'--allow-smoke\', action=\'store_true\')\n    args = p.parse_args()\n    print(json.dumps(verify(args.directory, args.allow_smoke), indent=2))\n', 'mm_eval.py': '"""\nMMFlood final test (pre-registered 9 Nov 2026). Same adaptation code and settings as ks_eval.py.\n\nPer event (chips from mm_prep.py) and model, store probability histograms for each method; the\nanalysis (mm_analyse.py) pools the parts and models and runs the pre-registered tests.\n\nUsage:  python mm_eval.py --data data --mm data_mm --runs runs --model-seed 0 --part 0 --nparts 2 --out runs_mm\n"""\nimport argparse\nimport csv\nimport os\nimport time\n\nos.environ.setdefault("GDAL_CACHEMAX", "64")\nimport numpy as np\n\nfrom ks_eval import terrain_10m\nfrom tta_px1 import BINS, curves, hand_channel, otsu_counts, run_one, save\n\nMETHODS_MM = ["none", "tent", "adabn", "adabn_phys", "adabn_physa"]\nMIN_CHIPS = 8\n\n\ndef load_mm_event(files):\n    ev = {"raw": [], "terr": [], "lab": [], "clean": []}\n    for f in files:\n        z = np.load(f)\n        ev["raw"].append(np.clip(np.stack([z["vv"], z["vh"]]).astype(np.float32), -50, 5))\n        hand = z["hand"]\n        ev["hand_ok"] = ev.get("hand_ok", 0) + int(np.any(hand != 0))\n        ev["terr"].append(hand_channel(hand, terrain_10m(z["dem"])))\n        ev["lab"].append(z["lab"])\n        ev["clean"].append(False)\n    return ev\n\n\ndef events_of(mm):\n    ev = {}\n    for r in csv.DictReader(open(os.path.join(mm, "index.csv"))):\n        ev.setdefault(r["event"], []).append(r["file"])\n    return {e: sorted(f) for e, f in sorted(ev.items()) if len(f) >= MIN_CHIPS}\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--data", default="data")\n    ap.add_argument("--mm", default="data_mm")\n    ap.add_argument("--runs", default="runs")\n    ap.add_argument("--out", default="runs_mm")\n    ap.add_argument("--model-seed", type=int, default=0)\n    ap.add_argument("--seed", type=int, default=0)\n    ap.add_argument("--steps", type=int, default=200)\n    ap.add_argument("--part", type=int, default=0)\n    ap.add_argument("--nparts", type=int, default=1)\n    ap.add_argument("--variants", default="V0")\n    ap.add_argument("--methods", default=",".join(METHODS_MM))\n    a = ap.parse_args()\n    a.methods, variants = a.methods.split(","), a.variants.split(",")\n\n    import torch\n    from sen1floods11_unet_baseline import channel_stats, read_split\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    os.makedirs(a.out, exist_ok=True)\n    path = os.path.join(a.out, f"mm_hist_m{a.model_seed}_p{a.part}.npz")\n    done = dict(np.load(path)) if os.path.exists(path) else {}\n    mean, std = channel_stats(a.data, read_split(a.data, "train"))\n    events = list(events_of(a.mm).items())[a.part::a.nparts]\n    print(f"part {a.part}/{a.nparts}: {len(events)} MMFlood events", flush=True)\n    for e, files in events:\n        todo = [(m, v) for v in variants for m in a.methods if f"{m}|{v}|{e}" not in done]\n        if not todo and f"otsu|{e}" in done:\n            continue\n        ev = load_mm_event(files)\n        if f"otsu|{e}" not in done:\n            done[f"otsu|{e}"], _ = otsu_counts(ev)\n        for m, v in todo:\n            t0 = time.time()\n            ckpt = os.path.join(a.runs, f"{v}_seed{a.model_seed}", "model.pt")\n            h = run_one(m, v, e, ev, mean, std, a, device, ckpt=ckpt)\n            done[f"{m}|{v}|{e}"] = h\n            fnr, _, iou = curves(h[0])\n            print(f"{m:11s} {v} event {e} ({len(files)} chips): miss rate at 0.5 {fnr[BINS // 2]:.3f}, IoU {iou[BINS // 2]:.3f}, "\n                  f"{time.time() - t0:.0f}s", flush=True)\n            save(path, done)\n        save(path, done)\n        if device == "cuda":\n            torch.cuda.empty_cache()\n    print("MM_EVAL_DONE", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'mm_prep.py': '"""\nMMFlood final-test preparation (pre-registered 9 Nov 2026; see the PX1 doc).\n\n  1. download mmflood.zip (Zenodo 6534637, 11.3 GB) and extract\n  2. drop events overlapping Kuro Siwo (300 km and 365 days) or Sen1Floods11 (300 km, same or adjacent year)\n  3. per image: dB conversion if linear, resample 20 m -> 10 m, cut 512 x 512 chips with >= 50% valid radar\n  4. at most 48 chips per event, spread evenly over the event\'s chips; events with < 8 chips dropped\n  5. labels: 1 water, 0 dry, 255 ignore (invalid radar; OSM permanent water that the mask calls dry)\n  6. HAND from GLO-30 HAND on each chip grid\n\nWrites data_mm/<event>/<chip>.npz (vv, vh dB float16; dem float32; hand float32; lab uint8),\ndata_mm/index.csv and data_mm/excluded.csv. Prints MM_PREP_DONE.\n"""\nimport argparse\nfrom contextlib import ExitStack\nimport csv\nimport glob\nimport json\nimport math\nimport os\nimport subprocess\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path, PurePosixPath\nfrom zipfile import ZipFile\n\nos.environ.setdefault("GDAL_CACHEMAX", "256")\nimport numpy as np\nimport rasterio\nfrom rasterio.transform import Affine\nfrom rasterio.warp import Resampling, reproject, transform_bounds\n\nURL = "https://zenodo.org/records/6534637/files/mmflood.zip?download=1"\nCHIP, MAX_CHIPS, MIN_CHIPS, MIN_VALID = 512, 48, 8, 0.5\nS1F_YEARS = {"Bolivia": 2018, "Ghana": 2018, "India": 2016, "Mekong": 2018, "Nigeria": 2018, "Pakistan": 2017,\n             "Paraguay": 2018, "Somalia": 2018, "Spain": 2019, "Sri-Lanka": 2017, "USA": 2019}\n\n\ndef km(lon1, lat1, lon2, lat2):\n    p1, p2 = math.radians(lat1), math.radians(lat2)\n    a = math.sin((p2 - p1) / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(math.radians(lon2 - lon1) / 2) ** 2\n    return 6371 * 2 * math.asin(math.sqrt(min(1, a)))\n\n\ndef s1f_centroids(data):\n    pts = {}\n    for f in glob.glob(os.path.join(data, "S1Hand", "*.tif")):\n        e = os.path.basename(f).split("_")[0]\n        with rasterio.open(f) as src:\n            b = src.bounds if src.crs.is_geographic else transform_bounds(src.crs, "EPSG:4326", *src.bounds)\n        pts.setdefault(e, []).append(((b[0] + b[2]) / 2, (b[1] + b[3]) / 2))\n    return {e: (np.mean([p[0] for p in v]), np.mean([p[1] for p in v])) for e, v in pts.items()}\n\n\ndef download(root):\n    z = os.path.join(root, "mmflood.zip")\n    if not os.path.exists(os.path.join(root, "extracted.ok")):\n        os.makedirs(root, exist_ok=True)\n        for t in range(4):\n            if subprocess.call(["wget", "-q", "-c", "-O", z, URL]) == 0:\n                break\n        print(f"downloaded {os.path.getsize(z) / 2**30:.1f} GB", flush=True)\n        subprocess.check_call(["unzip", "-q", "-o", z, "-d", root])\n        os.remove(z)\n        open(os.path.join(root, "extracted.ok"), "w").close()\n    for p in sorted(glob.glob(os.path.join(root, "*")) + glob.glob(os.path.join(root, "*", "*")))[:20]:\n        print(f"  layout: {p}", flush=True)\n    if not glob.glob(os.path.join(root, "**", "s1_raw"), recursive=True):  # archives inside the archive\n        for z in glob.glob(os.path.join(root, "**", "*.zip"), recursive=True) + glob.glob(os.path.join(root, "**", "*.tar*"), recursive=True):\n            print(f"  extracting inner archive {z}", flush=True)\n            if z.endswith(".zip"):\n                subprocess.call(["unzip", "-q", "-o", z, "-d", os.path.dirname(z)])\n            else:\n                subprocess.call(["tar", "-xf", z, "-C", os.path.dirname(z)])\n    acts = glob.glob(os.path.join(root, "**", "activations.json"), recursive=True)\n    # event folders = parents of every s1_raw folder, wherever the zip put them\n    dirs = {os.path.basename(os.path.dirname(p)): os.path.dirname(p)\n            for p in glob.glob(os.path.join(root, "**", "s1_raw"), recursive=True) if os.path.isdir(p)}\n    print(f"activations.json: {acts[:2]}; {len(dirs)} event folders", flush=True)\n    if not dirs:\n        raise SystemExit("no s1_raw folders found in the extracted archive")\n    return (acts[0] if acts else None), dirs\n\n\ndef to_db(a):\n    a = a.astype(np.float32)\n    v = a[np.isfinite(a) & (a != 0)]\n    if v.size and np.median(v) > 0:  # linear power -> dB\n        a = np.where(a > 0, 10 * np.log10(np.maximum(a, 1e-6)), np.nan)\n    return a\n\n\ndef resample(src, band, transform, crs, shape, method):\n    out = np.full(shape, np.nan, np.float32)\n    reproject(src.read(band).astype(np.float32), out, src_transform=src.transform, src_crs=src.crs,\n              dst_transform=transform, dst_crs=crs, src_nodata=src.nodata, dst_nodata=np.nan, resampling=method)\n    return out\n\n\ndef image_chips(base, name):\n    """All valid 512 x 512 chips of one MMFlood image, at 10 m."""\n    with rasterio.open(os.path.join(base, "s1_raw", name)) as s1:\n        crs, t = s1.crs, s1.transform\n        lat = (s1.bounds.top + s1.bounds.bottom) / 2\n        res_m = abs(t.a) * (111320 * math.cos(math.radians(lat)) if crs.is_geographic else 1)\n        f = max(1.0, res_m / 10.0)\n        shape = (int(s1.height * f), int(s1.width * f))\n        tt = t * Affine.scale(1 / f)\n        vv = to_db(resample(s1, 1, tt, crs, shape, Resampling.bilinear))\n        vh = to_db(resample(s1, 2, tt, crs, shape, Resampling.bilinear))\n    with rasterio.open(os.path.join(base, "mask", name)) as m:\n        mask = resample(m, 1, tt, crs, shape, Resampling.nearest)\n    with rasterio.open(os.path.join(base, "DEM", name)) as d:\n        dem = resample(d, 1, tt, crs, shape, Resampling.bilinear)\n    hp = os.path.join(base, "hydro", name)\n    hydro = None\n    if os.path.exists(hp):\n        with rasterio.open(hp) as h:\n            hydro = resample(h, 1, tt, crs, shape, Resampling.nearest)\n    valid = np.isfinite(vv) & np.isfinite(vh) & (vv > -49) & (vh > -49)\n    vals = set(np.unique(mask[np.isfinite(mask)]).astype(int).tolist())\n    water_val = 255 if vals <= {0, 255} and 255 in vals else 1  # masks stored as 0/255 or 0/1(/255 nodata)\n    lab = np.full(shape, 255, np.uint8)\n    lab[(mask == 0) & valid] = 0\n    lab[(mask == water_val) & valid] = 1\n    if hydro is not None:\n        lab[(hydro > 0) & (lab == 0)] = 255\n    chips = []\n    for r in range(0, shape[0] - CHIP + 1, CHIP):\n        for c in range(0, shape[1] - CHIP + 1, CHIP):\n            v = valid[r:r + CHIP, c:c + CHIP]\n            if v.mean() < MIN_VALID:\n                continue\n            d = dem[r:r + CHIP, c:c + CHIP]\n            d = np.nan_to_num(d, nan=float(np.nanmedian(d)) if np.isfinite(d).any() else 0.0)\n            chips.append({"vv": np.nan_to_num(vv[r:r + CHIP, c:c + CHIP], nan=-50.0), "vh": np.nan_to_num(vh[r:r + CHIP, c:c + CHIP], nan=-50.0),\n                          "dem": d, "lab": lab[r:r + CHIP, c:c + CHIP], "transform": tuple(tt * Affine.translation(c, r))[:6], "crs": crs.to_wkt()})\n    return chips\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--raw", default="mmflood_raw")\n    ap.add_argument("--out", default="data_mm")\n    ap.add_argument("--data", default="data")\n    ap.add_argument("--ks-events", default="ks_events.json")\n    ap.add_argument("--workers", type=int, default=4)\n    ap.add_argument("--smoke", action="store_true", help="Fetch only enough eligible regions to prepare one smoke region")\n    a = ap.parse_args()\n    with ExitStack() as stack:\n        prepare(a, stack)\n\n\ndef prepare(a, stack):\n    if a.smoke:\n        from mm_range_zip import RangeFile, read_member\n\n        source = stack.enter_context(RangeFile(URL))\n        archive = stack.enter_context(ZipFile(source))\n        members = set(archive.namelist())\n        radar = {}\n        for name in sorted(members):\n            path = PurePosixPath(name)\n            if path.suffix == \'.tif\' and path.parent.name == \'s1_raw\':\n                radar.setdefault(path.parent.parent.name, []).append(path)\n        dirs = {code: str(Path(a.raw) / paths[0].parent.parent) for code, paths in radar.items()}\n        meta_path = Path(a.raw) / \'activations.json\'\n        meta_path.parent.mkdir(parents=True, exist_ok=True)\n        meta_path.write_bytes(read_member(source, archive, \'activations.json\'))\n        print(f\'MMFlood smoke indexed {len(dirs)} regions; downloading eligible regions on demand\', flush=True)\n    else:\n        meta_path, dirs = download(a.raw)\n    meta = {}\n    if meta_path:\n        txt = open(meta_path).read()\n        try:\n            meta = json.loads(txt)\n        except json.JSONDecodeError:  # tolerate trailing commas\n            import re\n            meta = json.loads(re.sub(r",\\s*([}\\]])", r"\\1", txt))\n    if isinstance(meta, list):\n        meta = {m.get("code") or m.get("activation") or str(i): m for i, m in enumerate(meta)}\n    acts = {}\n    for d in sorted(dirs):\n        info = meta.get(d) or meta.get(d.split("-")[0]) or {}\n        if "lon" not in info or "lat" not in info:  # fall back to the first image\'s centre\n            if a.smoke:\n                # Use regions with published coordinates for the smoke diagnostic.\n                continue\n            tifs = sorted(glob.glob(os.path.join(dirs[d], "s1_raw", "*.tif")))\n            if tifs:\n                with rasterio.open(tifs[0]) as src:\n                    b = src.bounds if src.crs.is_geographic else transform_bounds(src.crs, "EPSG:4326", *src.bounds)\n                info = dict(info, lon=(b[0] + b[2]) / 2, lat=(b[1] + b[3]) / 2)\n        acts[d] = {"lon": info.get("lon", 0), "lat": info.get("lat", 0), "start": str(info.get("start") or info.get("end") or "2000-01-01"),\n                   "country": info.get("country"), "subset": info.get("subset")}\n    print(f"metadata example: {next(iter(meta.items()), None)}", flush=True)\n    ks = json.load(open(a.ks_events))\n    s1f = s1f_centroids(a.data)\n    keep, excl = [], []\n    for code, info in sorted(acts.items()):\n        lon, lat, date = float(info["lon"]), float(info["lat"]), info["start"][:10]\n        why = []\n        for k, e in ks.items():\n            if km(lon, lat, e["lon"], e["lat"]) <= 300:\n                dd = abs((np.datetime64(date) - np.datetime64(e["date"])).astype(int))\n                if dd <= 365:\n                    why.append(f"KuroSiwo {k} ({dd} days)")\n        for e, (elon, elat) in s1f.items():\n            if km(lon, lat, elon, elat) <= 300 and abs(int(date[:4]) - S1F_YEARS.get(e, 0)) <= 1:\n                why.append(f"Sen1Floods11 {e}")\n        (excl if why else keep).append((code, info, "; ".join(why)))\n    print(f"{len(acts)} activations; {len(excl)} excluded for overlap; {len(keep)} kept", flush=True)\n    os.makedirs(a.out, exist_ok=True)\n    with open(os.path.join(a.out, "excluded.csv"), "w", newline="") as f:\n        w = csv.writer(f)\n        w.writerow(["event", "country", "start", "reason"])\n        for code, info, why in excl:\n            w.writerow([code, info.get("country"), info["start"][:10], why])\n\n    import hand_fetch\n    hand_fetch.probe()\n    rows = []\n\n    def do_event(item):\n        code, info, _ = item\n        base = dirs[code]\n        if a.smoke:\n            print(f\'Fetching MMFlood smoke candidate {code}: {len(radar[code])} radar scenes\', flush=True)\n            for path in radar[code]:\n                for modality in (\'s1_raw\', \'DEM\', \'mask\', \'hydro\'):\n                    member = path.parent.parent / modality / path.name\n                    if modality == \'hydro\' and str(member) not in members:\n                        continue\n                    destination = Path(base) / modality / path.name\n                    destination.parent.mkdir(parents=True, exist_ok=True)\n                    destination.write_bytes(read_member(source, archive, str(member)))\n            print(f\'MMFlood smoke transferred {source.transferred / 1024**2:.1f} MiB from ZIP\', flush=True)\n        if not os.path.isdir(base):\n            return []\n        evdir = os.path.join(a.out, code)\n        if os.path.exists(os.path.join(evdir, "done.ok")):\n            return [r for r in csv.DictReader(open(os.path.join(evdir, "chips.csv")))]\n        chips = []\n        for name in sorted(os.listdir(os.path.join(base, "s1_raw"))):\n            try:\n                chips += [(name, ch) for ch in image_chips(base, name)]\n            except Exception as e:\n                print(f"  {code}/{name}: {type(e).__name__} {e}", flush=True)\n        if len(chips) < MIN_CHIPS:\n            return []\n        if len(chips) > MAX_CHIPS:\n            idx = np.linspace(0, len(chips) - 1, MAX_CHIPS).round().astype(int)\n            chips = [chips[i] for i in sorted(set(idx))]\n        os.makedirs(evdir, exist_ok=True)\n        out = []\n        for i, (name, ch) in enumerate(chips):\n            tr = Affine(*ch["transform"])\n            hand = hand_fetch.hand_on_grid(tr, rasterio.crs.CRS.from_wkt(ch["crs"]), (CHIP, CHIP))\n            fn = os.path.join(evdir, f"{i:03d}.npz")\n            np.savez_compressed(fn, vv=ch["vv"].astype(np.float16), vh=ch["vh"].astype(np.float16), dem=ch["dem"].astype(np.float32),\n                                hand=hand.astype(np.float32), lab=ch["lab"])\n            out.append({"event": code, "file": fn, "image": name, "country": info.get("country"), "start": info["start"][:10],\n                        "water_px": int((ch["lab"] == 1).sum()), "dry_px": int((ch["lab"] == 0).sum())})\n        with open(os.path.join(evdir, "chips.csv"), "w", newline="") as f:\n            w = csv.DictWriter(f, fieldnames=list(out[0].keys()))\n            w.writeheader(); w.writerows(out)\n        open(os.path.join(evdir, "done.ok"), "w").close()\n        print(f"  {code} ({info.get(\'country\')}): {len(out)} chips, water px {sum(r[\'water_px\'] for r in out)}, dry px {sum(r[\'dry_px\'] for r in out)}", flush=True)\n        return out\n\n    if a.smoke:\n        for candidate in keep:\n            rows = do_event(candidate)\n            if rows:\n                break\n        if not rows:\n            raise ValueError(\'No eligible MMFlood smoke region produced enough chips\')\n    else:\n        with ThreadPoolExecutor(a.workers) as ex:\n            for r in ex.map(do_event, keep):\n                rows += r\n    with open(os.path.join(a.out, "index.csv"), "w", newline="") as f:\n        w = csv.DictWriter(f, fieldnames=["event", "file", "image", "country", "start", "water_px", "dry_px"])\n        w.writeheader(); w.writerows(rows)\n    n_ev = len({r["event"] for r in rows})\n    print(f"MMFlood: {n_ev} events, {len(rows)} chips", flush=True)\n    print("MM_PREP_DONE", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'mm_range_zip.py': '"""Read selected members of the public MMFlood ZIP via bounded HTTP ranges."""\nimport io\nimport re\nimport struct\nimport zlib\nfrom zipfile import ZipFile\n\nimport requests\n\n\nclass RangeFile(io.RawIOBase):\n    def __init__(self, url):\n        self.url = url\n        self.position = 0\n        self.transferred = 0\n        self.size = self._fetch(0, 0)[1]\n\n    def _fetch(self, start, end):\n        expected = end - start + 1\n        for attempt in range(3):\n            try:\n                # Distinct cache keys prevent a proxy from returning a previous range.\n                with requests.get(self.url, params={\'px1_range\': f\'{start}-{end}\'},\n                                  headers={\'Range\': f\'bytes={start}-{end}\', \'Accept-Encoding\': \'identity\'},\n                                  stream=True, timeout=(10, 25)) as response:\n                    match = re.fullmatch(rf\'bytes {start}-{end}/(\\d+)\', response.headers.get(\'Content-Range\', \'\'))\n                    if response.status_code != 206 or not match:\n                        raise OSError(f\'Unexpected ZIP range response: HTTP {response.status_code}, \'\n                                      f\'{response.headers.get("Content-Range")}\')\n                    data = response.raw.read(expected + 1)\n                    if len(data) != expected:\n                        raise OSError(f\'Truncated or oversized ZIP range: {len(data)} != {expected}\')\n                    self.transferred += len(data)\n                    return data, int(match[1])\n            except (requests.RequestException, OSError) as exc:\n                if attempt == 2:\n                    raise OSError(f\'MMFlood ZIP range {start}-{end} failed\') from exc\n\n    def seekable(self):\n        return True\n\n    def tell(self):\n        return self.position\n\n    def seek(self, offset, whence=io.SEEK_SET):\n        base = {io.SEEK_SET: 0, io.SEEK_CUR: self.position, io.SEEK_END: self.size}[whence]\n        self.position = max(0, base + offset)\n        return self.position\n\n    def read(self, size=-1):\n        available = max(0, self.size - self.position)\n        size = available if size < 0 else min(size, available)\n        if size == 0:\n            return b\'\'\n        if size > 128 * 1024**2 or self.transferred + size > 512 * 1024**2:\n            raise OSError(\'MMFlood smoke download limit exceeded\')\n        data, total = self._fetch(self.position, self.position + size - 1)\n        if total != self.size:\n            raise OSError(\'MMFlood ZIP size changed during read\')\n        self.position += size\n        return data\n\n\ndef read_member(source, archive, name):\n    entry = archive.getinfo(name)\n    if entry.compress_type != 9:\n        return archive.read(entry)\n    import inflate64\n\n    source.seek(entry.header_offset)\n    header = source.read(30)\n    if header[:4] != b\'PK\\x03\\x04\' or entry.flag_bits & 1:\n        raise ValueError(f\'Unsupported ZIP member: {name}\')\n    filename_size, extra_size = struct.unpack_from(\'<HH\', header, 26)\n    source.seek(filename_size + extra_size, io.SEEK_CUR)\n    data = inflate64.Inflater().inflate(source.read(entry.compress_size))\n    if len(data) != entry.file_size or zlib.crc32(data) != entry.CRC:\n        raise ValueError(f\'ZIP member integrity check failed: {name}\')\n    return data\n\n\nif __name__ == \'__main__\':\n    import json\n    from mm_prep import URL\n    with RangeFile(URL) as source, ZipFile(source) as archive:\n        names = archive.namelist()\n        print(len(names), \'members; first entries:\', names[:12], flush=True)\n        print(\'activation metadata:\', [n for n in names if n.endswith(\'activations.json\')], flush=True)\n        print(\'first SAR files:\', [n for n in names if \'/s1_raw/\' in n and n.endswith(\'.tif\')][:5], flush=True)\n        print(\'ZIP compression methods:\', {entry.compress_type for entry in archive.infolist()}, flush=True)\n        meta = json.loads(read_member(source, archive, \'activations.json\'))\n        print(type(meta).__name__, str(meta)[:1200], flush=True)\n', 'ks_events.json': '{\n"118": {\n"lon": -2.333,\n"lat": 42.358,\n"date": "2015-02-02"\n},\n"130": {\n"lon": 94.896,\n"lat": 17.072,\n"date": "2015-08-01"\n},\n"147": {\n"lon": -2.821,\n"lat": 54.647,\n"date": "2015-12-05"\n},\n"267": {\n"lon": 21.491,\n"lat": 55.287,\n"date": "2018-01-31"\n},\n"273": {\n"lon": 19.546,\n"lat": 42.01,\n"date": "2018-03-10"\n},\n"275": {\n"lon": 16.674,\n"lat": 45.387,\n"date": "2018-03-21"\n},\n"277": {\n"lon": 26.462,\n"lat": 41.291,\n"date": "2018-03-27"\n},\n"279": {\n"lon": -0.248,\n"lat": 41.203,\n"date": "2018-04-12"\n},\n"321": {\n"lon": -86.991,\n"lat": 13.049,\n"date": "2018-10-06"\n},\n"324": {\n"lon": 2.674,\n"lat": 43.226,\n"date": "2018-10-15"\n},\n"417": {\n"lon": -8.638,\n"lat": 40.189,\n"date": "2019-12-19"\n},\n"421": {\n"lon": 2.96,\n"lat": 42.721,\n"date": "2020-01-22"\n},\n"427": {\n"lon": 13.141,\n"lat": 58.173,\n"date": "2020-02-18"\n},\n"437": {\n"lon": -0.093,\n"lat": 43.971,\n"date": "2020-05-10"\n},\n"445": {\n"lon": 26.705,\n"lat": 48.199,\n"date": "2020-06-26"\n},\n"470": {\n"lon": 0.573,\n"lat": 10.409,\n"date": "2020-10-05"\n},\n"492": {\n"lon": -0.982,\n"lat": 43.705,\n"date": "2021-01-01"\n},\n"497": {\n"lon": 6.651,\n"lat": 51.446,\n"date": "2021-02-01"\n},\n"498": {\n"lon": 1.842,\n"lat": 45.011,\n"date": "2021-02-02"\n},\n"502": {\n"lon": -8.449,\n"lat": 52.029,\n"date": "2021-02-23"\n},\n"518": {\n"lon": 5.632,\n"lat": 50.571,\n"date": "2021-07-14"\n},\n"555": {\n"lon": -1.848,\n"lat": 42.449,\n"date": "2021-12-10"\n},\n"567": {\n"lon": 152.999,\n"lat": -29.618,\n"date": "2022-02-26"\n},\n"1111004": {\n"lon": -96.354,\n"lat": 29.119,\n"date": "2017-08-28"\n},\n"1111005": {\n"lon": 48.402,\n"lat": -17.628,\n"date": "2020-01-25"\n},\n"1111006": {\n"lon": 46.6,\n"lat": -16.461,\n"date": "2020-01-30"\n},\n"1111007": {\n"lon": 85.67,\n"lat": 25.55,\n"date": "2019-09-17"\n},\n"1111008": {\n"lon": -83.538,\n"lat": 14.072,\n"date": "2020-08-15"\n},\n"1111009": {\n"lon": 68.295,\n"lat": 27.36,\n"date": "2022-09-10"\n},\n"1111010": {\n"lon": -80.712,\n"lat": -5.568,\n"date": "2017-03-10"\n},\n"1111011": {\n"lon": 121.698,\n"lat": 17.584,\n"date": "2020-11-12"\n},\n"1111013": {\n"lon": -90.109,\n"lat": 38.386,\n"date": "2017-05-03"\n},\n"554": {\n"lon": 151.171,\n"lat": -28.431,\n"date": "2021-11-30"\n}\n}', 'run_kaggle.py': '"""\nPX1 on Kaggle: one entry point for every remaining experiment.\n\n  python run_kaggle.py --task physa      adaptive physics rule (per-flood radar thresholds + HAND)\n                                         on Sen1Floods11 (11 held-out floods) and Kuro Siwo (3 models x 26 events)\n\nLayout (everything big lives in /tmp, only results go to /kaggle/working/out):\n  bundle  = the uploaded dataset (code, model weights, PC histograms), found anywhere under /kaggle/input\n  WORK    = /tmp/px1   code + data + runs\n  OUT     = /kaggle/working/out   logs, histograms, summaries (the notebook\'s saved output)\n\nData are downloaded here, not uploaded: Sen1Floods11 hand-labelled chips (Google Cloud), the Copernicus DEM and\nGLO-30 HAND (AWS), and the same evenly spaced Kuro Siwo sample as on the PC (Hugging Face, step 512 MB, 6 per window).\n"""\nimport argparse\nimport glob\nimport os\nimport shutil\nimport subprocess\nimport sys\nimport threading\nimport time\n\nWORK, OUT = "/tmp/px1", "/kaggle/working/out"\n\n\ndef sh(cmd, log=None, env=None, cwd=WORK):\n    print(f"$ {cmd}", flush=True)\n    if log:\n        with open(log, "a") as f:\n            return subprocess.call(cmd, shell=True, stdout=f, stderr=subprocess.STDOUT, env=env, cwd=cwd)\n    return subprocess.call(cmd, shell=True, env=env, cwd=cwd)\n\n\nROOTS = ["/kaggle/input", "/tmp/px1_bundle"]\n\n\ndef unpack_zips():\n    """Kaggle may keep uploaded zips packed: extract any px1 zips into /tmp/px1_bundle."""\n    import zipfile\n    for z in glob.glob("/kaggle/input/**/px1_part*.zip", recursive=True):\n        dst = os.path.join("/tmp/px1_bundle", os.path.basename(z)[:-4])\n        if not os.path.exists(dst):\n            print(f"extracting {z}", flush=True)\n            with zipfile.ZipFile(z) as f:\n                f.extractall(dst)\n\n\ndef rglob(pattern):\n    return [p for r in ROOTS for p in glob.glob(os.path.join(r, "**", pattern), recursive=True)]\n\n\ndef find_bundle():\n    unpack_zips()\n    hits = rglob("run_kaggle.py")\n    if not hits:\n        raise SystemExit("bundle not found under /kaggle/input (add the px1 dataset to the notebook)")\n    return os.path.dirname(hits[0])\n\n\ndef setup(task="physa"):\n    t0 = time.time()\n    os.makedirs(WORK, exist_ok=True)\n    os.makedirs(os.path.join(OUT, "logs"), exist_ok=True)\n    sh(f"{sys.executable} -m pip install -q rasterio", cwd="/tmp")\n    code = find_bundle()\n    for f in glob.glob(os.path.join(code, "*.py")):\n        shutil.copy(f, WORK)\n    for f in glob.glob("/tmp/px1_override/*"):  # newer code typed into the notebook overrides the bundle\n        shutil.copy(f, WORK)\n        print(f"override: {os.path.basename(f)}", flush=True)\n    # model weights and PC histograms, wherever the zips were extracted\n    for p in rglob("model.pt"):\n        run, group = os.path.basename(os.path.dirname(p)), os.path.basename(os.path.dirname(os.path.dirname(p)))\n        if group in ("runs", "runs_loeo"):\n            d = os.path.join(WORK, group, run)\n            os.makedirs(d, exist_ok=True)\n            shutil.copy(p, os.path.join(d, "model.pt"))\n    for p in rglob("clean_chips.txt"):\n        os.makedirs(os.path.join(WORK, "audit"), exist_ok=True)\n        shutil.copy(p, os.path.join(WORK, "audit", "clean_chips.txt"))\n    for p in [p for p in rglob("*.npz") if os.path.basename(os.path.dirname(p)) == "pc_hist"]:\n        name = os.path.basename(p)\n        dst = os.path.join(WORK, "runs_loeo" if name.startswith("tta_hist") else "runs_ks", name.replace(".npz", "_local.npz"))\n        os.makedirs(os.path.dirname(dst), exist_ok=True)\n        shutil.copy(p, dst)\n    n_models = len(glob.glob(os.path.join(WORK, "runs*", "*", "model.pt")))\n    print(f"bundle: {code}; {n_models} models", flush=True)\n\n    L = os.path.join(OUT, "logs", "setup.txt")\n    py = sys.executable\n    steps = [f"{py} -u download_sen1floods11.py --out data --workers 16"]\n    if task == "physa":\n        steps += [f"{py} -u dem_fetch.py --data data --workers 16",\n                  f"{py} -u ks_download.py --splits test_GRD train_GRD --step-mb 512 --k 6",\n                  f"{py} -u hand_fetch.py --data data --ks data_ks --workers 16"]\n    if task == "mmflood":\n        steps += [f"{py} -u mm_prep.py --raw mmflood_raw --out data_mm --data data --ks-events ks_events.json --workers 4"]\n    for s in steps:\n        for attempt in range(3):\n            if sh(s, log=L) == 0:\n                break\n            print(f"  retry {attempt + 1}: {s}", flush=True)\n        else:\n            raise SystemExit(f"setup step failed: {s} (see {L})")\n    print(f"setup done in {(time.time() - t0) / 60:.0f} min", flush=True)\n\n\ndef run_parallel(jobs, per_gpu=2):\n    """jobs: list of (name, command). Runs per_gpu jobs at a time on each visible GPU."""\n    import torch\n    n_gpu = max(1, torch.cuda.device_count())\n    # each GPU gets its own queue, jobs dealt round-robin, so no GPU sits idle while another has two jobs\n    queues = {g: [j for i, j in enumerate(jobs) if i % n_gpu == g] for g in range(n_gpu)}\n    slots = [g for g in range(n_gpu) for _ in range(per_gpu)]\n    lock, status = threading.Lock(), {}\n\n    def worker(gpu):\n        while True:\n            with lock:\n                if not queues[gpu]:\n                    return\n                name, cmd = queues[gpu].pop(0)\n            env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))\n            status[name] = "running"\n            rc = sh(cmd, log=os.path.join(OUT, "logs", f"{name}.txt"), env=env)\n            if rc != 0:  # resume once (the scripts skip finished work)\n                rc = sh(cmd, log=os.path.join(OUT, "logs", f"{name}.txt"), env=env)\n            status[name] = f"exit {rc}"\n\n    threads = [threading.Thread(target=worker, args=(g,)) for g in slots]\n    for t in threads:\n        t.start()\n    while any(t.is_alive() for t in threads):\n        time.sleep(300)\n        for name, _ in jobs:\n            log = os.path.join(OUT, "logs", f"{name}.txt")\n            last = ""\n            if os.path.exists(log):\n                with open(log) as f:\n                    tail = [ln for ln in f.read().splitlines() if "event" in ln or "hold" in ln]\n                last = tail[-1][:90] if tail else ""\n            print(f"[{time.strftime(\'%H:%M\')}] {name}: {status.get(name, \'queued\')} | {last}", flush=True)\n    for t in threads:\n        t.join()\n    return status\n\n\ndef task_physa():\n    py = sys.executable\n    m = "none,adabn_phys,physa,adabn_physa"  # none and adabn_phys re-computed as a reproducibility check\n    jobs = [(f"ks_m{s}", f"{py} -u ks_eval.py --data data --ks data_ks --runs runs --model-seed {s} --out runs_ks --methods {m}")\n            for s in range(3)]\n    jobs.append(("s1f", f"{py} -u tta_px1.py --data data --runs runs_loeo --seed 0 --model-seed 0 --methods {m}"))\n    status = run_parallel(jobs)\n    print(status, flush=True)\n    for p in glob.glob(os.path.join(WORK, "runs_loeo", "tta_*")) + glob.glob(os.path.join(WORK, "runs_ks", "ks_*")):\n        shutil.copy(p, OUT)\n    sh(f"{py} -u merge_analyse.py runs_loeo runs_ks {OUT}")\n\n\ndef task_mmflood(primary):\n    py = sys.executable\n    jobs = [(f"mm_m{s}_p{p}", f"{py} -u mm_eval.py --data data --mm data_mm --runs runs --model-seed {s} --part {p} --nparts 2 --out runs_mm")\n            for p in range(2) for s in range(3)]\n    status = run_parallel(jobs, per_gpu=3)\n    print(status, flush=True)\n    for p in glob.glob(os.path.join(WORK, "runs_mm", "*")) + glob.glob(os.path.join(WORK, "data_mm", "*.csv")):\n        shutil.copy(p, OUT)\n    sh(f"{py} -u mm_analyse.py --primary {primary} --runs runs_mm --mm data_mm --out {OUT}")\n\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--task", required=True, choices=["setup", "physa", "mmflood"])\n    ap.add_argument("--primary", default="adabn_physa", choices=["adabn_physa", "adabn_phys"])\n    a = ap.parse_args()\n    setup(a.task)\n    if a.task == "physa":\n        task_physa()\n    if a.task == "mmflood":\n        task_mmflood(a.primary)\n    print("KAGGLE_TASK_DONE", flush=True)\n'}

import os, sys, json, hashlib, subprocess, time, uuid, shutil
from pathlib import Path
import torch
if not Path('/kaggle/input').is_dir() or not torch.cuda.is_available():
    raise RuntimeError('Run this notebook on Kaggle GPU; it is not a PC training launcher')
START = time.monotonic()
RUN_ID = time.strftime('%Y%m%dT%H%M%SZ', time.gmtime()) + '_' + uuid.uuid4().hex[:8]
OUT = Path('/kaggle/working/baselines_v2') / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)
ST = OUT / 'code'
ST.mkdir()
for name, content in FILES.items():
    (ST / name).write_text(content, encoding='utf-8')
fingerprints = {name: hashlib.sha256(content.encode()).hexdigest() for name, content in FILES.items()}
manifest = dict(schema='px1-baselines-v2-run', dataset=DATASET, mode=MODE, status='started',
                run_id=RUN_ID, files=fingerprints, jobs=[], cuda=torch.version.cuda, torch=torch.__version__)
def record():
    (OUT / 'run_manifest.json').write_text(json.dumps(manifest, indent=2))
record()
W = Path('/tmp/px1')
active = []
logs = []
def execute_group(jobs):
    global active, logs
    active, logs = [], []
    try:
        for name, args, gpu in jobs:
            log = open(OUT / (name + '.log'), 'w')
            logs.append(log)
            command = [sys.executable, '-u', str(W / 'eval_baselines_v2.py'), '--dataset', DATASET,
                       '--work', str(W), '--out', str(OUT / name)] + args
            proc = subprocess.Popen(command, cwd=W, env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu),
                OMP_NUM_THREADS='1', MKL_NUM_THREADS='1'), stdout=log, stderr=subprocess.STDOUT)
            active.append((name, proc))
        while any(proc.poll() is None for _, proc in active):
            failures = [(name, proc.returncode) for name, proc in active if proc.poll() not in (None, 0)]
            if failures:
                raise RuntimeError(f'Failed jobs: {failures}')
            if time.monotonic() - START > 10.5 * 3600:
                raise TimeoutError('Notebook budget exceeded; partial results retained')
            time.sleep(10)
        failures = [(name, proc.returncode) for name, proc in active if proc.returncode != 0]
        if failures:
            raise RuntimeError(f'Failed jobs: {failures}')
        for name, _ in active:
            child = json.loads((OUT / name / 'manifest.json').read_text())
            if child['status'] != 'completed' or len(child['records']) != len(child['events']):
                raise RuntimeError(f'Incomplete output: {name}')
            verify_command = [sys.executable, str(W / 'verify_baseline_outputs.py'), str(OUT / name)]
            if name.startswith('smoke'):
                verify_command.append('--allow-smoke')
            subprocess.run(verify_command, cwd=W, check=True)
            manifest['jobs'].append(dict(name=name, status='completed'))
            print('Completed', name, flush=True)
        record()
    finally:
        for _, proc in active:
            if proc.poll() is None:
                proc.terminate()
                try:
                    proc.wait(timeout=15)
                except subprocess.TimeoutExpired:
                    proc.kill()
                    proc.wait()
        for log in logs:
            log.close()

try:
    subprocess.run([sys.executable, str(ST / 'test_baselines_v2.py')], cwd=ST, check=True,
                   env=dict(os.environ, CUDA_VISIBLE_DEVICES='', OMP_NUM_THREADS='1', MKL_NUM_THREADS='1'))
    if W.exists() or Path('/tmp/px1_override').exists():
        raise RuntimeError('Use a fresh Kaggle session: refusing to overwrite a prior PX1 workspace')
    override = Path('/tmp/px1_override')
    override.mkdir()
    for name in FILES:
        shutil.copy2(ST / name, override / name)
    sys.path.insert(0, str(ST))
    import run_kaggle
    run_kaggle.OUT = str(OUT / 'setup')
    # Only existing project datasets. S1F additionally needs terrain for proposed-method controls.
    if DATASET == 'ks':
        run_kaggle.setup('physa')
    elif DATASET == 'mm' and MODE == 'smoke':
        run_kaggle.setup('setup')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'inflate64==1.0.4'], cwd=W, check=True)
        with (OUT / 'setup/logs/mm_smoke.txt').open('w') as log:
            subprocess.run([sys.executable, '-u', 'mm_prep.py', '--smoke'], cwd=W,
                           stdout=log, stderr=subprocess.STDOUT, check=True)
    elif DATASET == 'mm':
        run_kaggle.setup('mmflood')
    else:
        run_kaggle.setup('setup')
        subprocess.run([sys.executable, 'dem_fetch.py', '--data', 'data', '--workers', '8'], cwd=W, check=True)
        subprocess.run([sys.executable, 'hand_fetch.py', '--data', 'data', '--sen1floods11', '--workers', '8'], cwd=W, check=True)
    if DATASET == 'ks':
        # setup('physa') also prepares source terrain; no future dataset access.
        pass
    # Freeze exact runtime packages for this notebook output, not just requested requirements.
    freeze = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
    (OUT / 'environment.txt').write_text(freeze)
    execute_group([('smoke_seed0', ['--seed', '0', '--smoke'], 0)])
    if MODE == 'full':
        seeds = [0] if DATASET == 's1f' else [0, 1, 2]
        devices = torch.cuda.device_count()
        for offset in range(0, len(seeds), devices):
            execute_group([(f'full_seed{s}', ['--seed', str(s)], gpu)
                           for gpu, s in enumerate(seeds[offset:offset+devices])])
    manifest['status'] = 'completed'
except BaseException as exc:
    manifest['status'], manifest['error'] = 'failed', repr(exc)
    raise
finally:
    manifest['seconds'] = time.monotonic() - START
    record()
print('BASELINES_V2_NOTEBOOK_DONE', OUT, flush=True)
